In [2]:
import pandas as pd

# This loads ALL sheets at once into a dictionary
all_sheets = pd.read_excel(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\Version 1.4 -- Ghana Health Service Data Project - Copy.xlsx", sheet_name=None)

In [3]:
# See all the sheet names
print(all_sheets.keys())

dict_keys(['raw data', 'hospital-to-centroids-km', 'hospitals-data', 'Table2', 'all-districts-data', 'gss-population-distribution', 'district-centroids', 'not-relevant-yet', 'analysis'])


In [4]:
# See each sheet name and its size
for name, df in all_sheets.items():
    print(f"{name}: {df.shape[0]} rows x {df.shape[1]} columns")

raw data: 10337 rows x 24 columns
hospital-to-centroids-km: 10337 rows x 21 columns
hospitals-data: 9978 rows x 13 columns
Table2: 834 rows x 17 columns
all-districts-data: 834 rows x 17 columns
gss-population-distribution: 261 rows x 6 columns
district-centroids: 261 rows x 4 columns
not-relevant-yet: 273 rows x 7 columns
analysis: 264 rows x 5 columns


In [6]:
pip install fuzzywuzzy python-Levenshtein

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.5 MB ? eta -:--:--
   ------------- -------------------------- 0.5/1.5 MB 1.8 MB/s eta 0:00:01
   -------------------- ------------------- 0.8/1.5 MB 1.5 MB/s eta 0:00:01
   --------------------------- ------------ 1.0/1.5 MB 1.1 MB/s eta 0:00:01
   ---------------------------------- ----- 1.3/1.5 MB 1.1 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 1.1 MB/s eta 0:00:00

   ---------- ----------------------------- 1/4 [rapidfuzz]
   ---------- ----------------------------- 1/4 [rapidfuzz]
   ---------- ----------------------------- 1/4 [rapidfuzz]
   -------------------- ------------------- 2/4 [Levenshtein]
   ---------------------------------------- 4/4 [python-Levenshtein]

Note: you may need to restart the kernel to use updated packages.


In [10]:
# Step 1: Let's build the mapping from hospital district names to population district names
# We'll use fuzzy matching to find the closest match

# First install fuzzywuzzy if you don't have it
# pip install fuzzywuzzy python-Levenshtein

from fuzzywuzzy import fuzz, process
import pandas as pd

# Load the datasets
all_sheets = pd.read_excel(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\Version 1.4 -- Ghana Health Service Data Project - Copy.xlsx", sheet_name=None)

hospitals = all_sheets['hospitals-data']
population = all_sheets['gss-population-distribution']

# Get unique district names from each
hospital_districts = sorted(hospitals['District'].str.strip().str.upper().unique())
population_districts = sorted(population['District'].str.strip().str.upper().unique())

# Find the best match in population for each hospital district
mapping = []

for h_dist in hospital_districts:
    # Check if it already matches exactly
    if h_dist in population_districts:
        mapping.append({
            'hospital_district': h_dist,
            'population_district': h_dist,
            'match_score': 100,
            'status': 'EXACT'
        })
    else:
        # Find the closest fuzzy match
        best_match, score = process.extractOne(h_dist, population_districts, scorer=fuzz.token_sort_ratio)
        mapping.append({
            'hospital_district': h_dist,
            'population_district': best_match,
            'match_score': score,
            'status': 'FUZZY'
        })

mapping_df = pd.DataFrame(mapping)

# Show only the fuzzy matches so you can verify them
fuzzy_matches = mapping_df[mapping_df['status'] == 'FUZZY'].sort_values('match_score')

print(f"Total districts: {len(mapping_df)}")
print(f"Exact matches: {len(mapping_df[mapping_df['status'] == 'EXACT'])}")
print(f"Fuzzy matches that need your review: {len(fuzzy_matches)}")
print(f"\n{'='*90}")
print(f"{'HOSPITAL NAME':<45} {'POPULATION NAME':<35} {'SCORE'}")
print(f"{'='*90}")

for _, row in fuzzy_matches.iterrows():
    print(f"{row['hospital_district']:<45} {row['population_district']:<35} {row['match_score']}")

Total districts: 261
Exact matches: 157
Fuzzy matches that need your review: 104

HOSPITAL NAME                                 POPULATION NAME                     SCORE
HO                                            HO-WEST                             44
EJISU                                         PRU EAST                            46
EFUTU                                         EFFUTU MUNICIPAL                    48
HOHOE                                         HO-WEST                             50
YENDI                                         BINDURI                             50
MION                                          TAIN                                50
BEKWAI                                        BEKWAI MUNICIPAL                    55
KETA                                          GA EAST                             55
WENCHI                                        WENCHI MUNICIPAL                    55
GUSHIEGU                                      PUSIGA             

In [9]:
# Manual mapping for all 104 fuzzy matches
# Hospital district name -> Population district name

district_mapping = {
    # Correct fuzzy matches (verified)
    'AFADJATO SOUTH': 'AFADZATO SOUTH',
    'AJUMAKO-ENYAN-ESSIAM': 'AJUMAKU ENYAN ESSIAM',
    'ABURA-ASEBU-KWAMANKESE': 'ABURA ASEBU KWAMANKESE',
    'ASIKUMA-ODOBEN-BRAKWA': 'ASIKUMA ODOBEN BRAKWA',
    'ATEBUBU-AMANTEN': 'ATEBUBU AMANTIN',
    'BUNKPURUGU -NAKPANDURI': 'BUNKPURUGU NYANKPANDURI',
    'KASENA-NANKANA WEST': 'KASSENA NANKANA WEST',
    'TWIFO-HEMANG LOWER DENKYIRA': 'TWIFO HEMAN LOWER DENKYIRA',
    'HO WEST': 'HO-WEST',
    'KPONE-KATAMANSO': 'KPONE KATAMANSO',
    'KORLE-KLOTTEY': 'KORLE KLOTTEY',
    'SEFWI-AKONTOMBRA': 'SEFWI AKONTOMBRA',
    'PRESTEA-HUNI VALLEY': 'PRESTEA/HUNI VALLEY',
    'MAMPRUGU-MOAGDURI': 'MAMPRUGU MOAGDURI',
    'SHAI-OSUDOKU': 'SHAI OSUDOKU',
    'NORTH-EAST GONJA': 'NORTH EAST GONJA',
    'UPPER MANYA-KROBO': 'UPPER MANYA KROBO',
    'BOSOMTWE': 'BOSOMTWI',
    'NSAWAM-ADOAGYIRI': 'NSAWAM ADOAGYIRI MUNICIPAL',
    'LOWER MANYA-KROBO': 'LOWER MANYA KROBO MUNICIPAL',
    'EJURA-SEKYEDUMASE': 'EJURA SEKYEDUMASE MUNICIPAL',
    'LA-NKWANTANANG-MADINA': 'LA NKWANTANAN-MADINA MUNICIPAL',
    'EFFIA-KWESIMINTSIM': 'EFFIA KWESIMINTSIM MUNICIPAL',
    'ASANTE AKIM CENTRAL': 'ASANTE AKIM CENTRAL MUNICIPAL',
    'UPPER DENKYIRA EAST': 'UPPER DENKYIRA EAST MUNICIPAL',
    'KOMENDA-EDNA-EGUAFO-ABIREM': 'KOMENDA EDINA EGUAFO ABIREM MUNICIPAL',
    'DAFFIAMA-BUSSIE-ISSA': 'DAFFIAMA BUSSIE',
    'BIBIANI-ANHWIASO-BEKWAI': 'SEFWI BIBIANI AHWIASO BEKWAI',
    'ASOKORE MAMPONG': 'ASOKORE MAMPONG MUNICIPAL',
    'AWUTU SENYA EAST': 'AWUTU SENYA EAST MUNICIPAL',
    'ABLEKUMA NORTH': 'ABLEKUMA NORTH MUNICIPAL',
    'ABLEKUMA CENTRAL': 'ABLEKUMA CENTRAL MUNICIPAL',
    'AYAWASO CENTRAL': 'AYAWASO CENTRAL MUNICIPAL',
    'LAMBUSSIE': 'LAMBUSSIE-KARNI',
    'WEIJA-GBAWE': 'WEIJA GBAWE MUNICIPAL',
    'YILO-KROBO': 'YILO KROBO MUNICIPAL',
    'OFORIKROM': 'OFORIKROM MUNICIPAL',
    'SAGNARIGU': 'SAGNARIGU MUNICIPAL',
    'SAVELUGU': 'SAVELUGU MUNICIPAL',
    'OLD TAFO': 'OLD TAFO MUNICIPAL',
    'MPOHOR': 'MOPHOR (MPOHOR)',
    'BIRIM CENTRAL': 'BIRIM CENTRAL MUNICIPAL',
    'LA-DADE-KOTOPON': 'LA DADEKOTOPON MUNICIPAL',
    'TARKWA-NSUAEM': 'TARKWA-NSUAEM MUNICIPAL',
    'SEKONDI-TAKORADI': 'SEKONDI TAKORADI METROPOLITAN',
    'SEFWI-WIAWSO': 'SEFWI WIAWSO MUNICIPAL',
    'SUNYANI WEST': 'SUNYANI WEST MUNICIPAL',
    'ADANSI AKROFUOM': 'AKROFUOM',
    'NZEMA EAST': 'NZEMA EAST MUNICIPAL',
    'TATALE-SANGULE': 'TATALE',
    'MFANTSEMAN': 'MFANTSIMAN MUNICIPAL',
    'CAPE COAST': 'CAPE COAST METROPOLITAN',
    'ACCRA METRO': 'ACCRA METROPOLITAN',
    'BEKWAI': 'BEKWAI MUNICIPAL',
    'WENCHI': 'WENCHI MUNICIPAL',
    
    # Matches that need the MUNICIPAL/METROPOLITAN suffix
    'HO': 'HO MUNICIPAL',
    'EJISU': 'EJISU JUABEN MUNICIPAL',
    'EFUTU': 'EFFUTU MUNICIPAL',
    'HOHOE': 'HOHOE MUNICIPAL',
    'YENDI': 'YENDI MUNICIPAL',
    'MION': 'MION DISTRICT',
    'KETA': 'KETA MUNICIPAL',
    'GUSHIEGU': 'GUSHEGU MUNICIPAL',
    'ASANTE MAMPONG': 'MAMPONG MUNICIPAL',
    'KWADASO': 'KWADASO MUNICIPAL',
    'SUHUM': 'SUHUM MUNICIPAL',
    'ADENTAN': 'ADENTAN MUNICIPAL',
    'JUABEN': 'JUABEN MUNICIPAL',
    'ASOKWA': 'ASOKWA MUNICIPAL',
    'KUMASI': 'KUMASI METROPOLITAN',
    'AOWIN': 'AOWIN MUNICIPAL',
    'TEMA': 'TEMA METROPOLITAN',
    'SUAME': 'SUAME MUNICIPAL',
    'BEREKUM': 'BEREKUM EAST MUNICIPAL',
    'OBUASI': 'OBUASI MUNICIPAL',
    'OFFINSO': 'OFFINSO MUNICIPAL',
    'TAMALE': 'TAMALE METROPOLITAN',
    'DORMAA MUNICIPAL': 'DORMAA CENTRAL MUNICIPAL',
    'GA CENTRAL': 'GA CENTRAL MUNICIPAL',
    'NEW JUABEN SOUTH': 'NEW JUABEN SOUTH MUNICIPAL',
    'AGONA WEST': 'AGONA WEST MUNICIPAL',
    'EAST MAMPRUSI': 'EAST MAMPRUSI MUNICIPAL',
    'WEST MAMPRUSI': 'WEST MAMPRUSI MUNICIPAL',
    'NANUMBA NORTH': 'NANUMBA NORTH MUNICIPAL',
    'KINTAMPO NORTH': 'KINTAMPO NORTH MUNICIPAL',
    'NKORANZA SOUTH': 'NKORANZA SOUTH MUNICIPAL',
    'BUILSA NORTH': 'BUILSA NORTH MUNICIPAL',
    'ASUNAFO NORTH': 'ASUNAFO NORTH MUNICIPAL',
    'AKWAPIM NORTH': 'AKWAPIM NORTH MUNICIPAL',
    'ABUAKWA SOUTH': 'ABUAKWA SOUTH MUNICIPAL',
    'JAMAN SOUTH': 'JAMAN SOUTH MUNICIPAL',
    'KPANDO': 'KPANDO MUNICIPAL',
    'KWAHU WEST': 'KWAHU WEST MUNICIPAL',
    'WEST AKIM': 'WEST AKIM MUNICIPAL',
    'ABLEKUMA WEST': 'ABLEKUMA WEST MUNICIPAL',
    'AYAWASO EAST': 'AYAWASO EAST MUNICIPAL',
    'AYAWASO WEST': 'AYAWASO WEST MUNICIPAL',
    'AYAWASO NORTH': 'AYAWASO NORTH MUNICIPAL',
    'TANO NORTH': 'TANO NORTH MUNICIPAL',
    'TANO SOUTH': 'TANO SOUTH MUNICIPAL',
    'KASENA-NANKANA': 'KASSENA NANKANA EAST MUNICIPAL',
    'ASSIN CENTRAL': 'ASSIN CENTRAL MUNICIPAL',
    'ATWIMA NWABIAGYA': 'ATWIMA NWABIAGYA SOUTH MUNICIPAL',
    'AWUTU SENYA': 'AWUTU SENYA WEST',
}

# Now verify: check that every population name in the mapping actually exists
pop_districts_upper = set(population['District'].str.strip().str.upper())

print("VERIFYING ALL MAPPINGS...")
print("="*70)

bad_mappings = []
for hosp, pop in district_mapping.items():
    if pop.upper() not in pop_districts_upper:
        bad_mappings.append((hosp, pop))

if bad_mappings:
    print(f"\n❌ {len(bad_mappings)} mappings point to population names that DON'T EXIST:")
    for h, p in bad_mappings:
        print(f"  {h} -> {p} (NOT FOUND)")
else:
    print("✅ All mappings verified! Every population name exists.")

# Check if we've covered all 104 fuzzy matches
print(f"\nMappings created: {len(district_mapping)}")
print(f"Fuzzy matches needed: 104")

if len(district_mapping) == 104:
    print("✅ All 104 districts covered!")
else:
    print(f"⚠️  We have {len(district_mapping)} mappings but need 104")

VERIFYING ALL MAPPINGS...
✅ All mappings verified! Every population name exists.

Mappings created: 104
Fuzzy matches needed: 104
✅ All 104 districts covered!


In [11]:
# Step 1: Create a standardized district column in hospitals data
# that uses the population dataset's naming convention

hospitals_clean = hospitals.copy()

# Make hospital districts uppercase for matching
hospitals_clean['District_upper'] = hospitals_clean['District'].str.strip().str.upper()

# Apply the mapping — if a district is in our translation book, use the population name
# If not, it's already an exact match so keep it as is
hospitals_clean['District_standardized'] = hospitals_clean['District_upper'].map(
    lambda x: district_mapping.get(x, x)
)

# Now let's verify: do ALL hospital districts now match a population district?
pop_districts_upper = set(population['District'].str.strip().str.upper())
hosp_standardized = set(hospitals_clean['District_standardized'].str.upper())

matched = hosp_standardized & pop_districts_upper
unmatched = hosp_standardized - pop_districts_upper

print(f"Total hospital districts: {len(hosp_standardized)}")
print(f"Matched to population: {len(matched)}")
print(f"Still unmatched: {len(unmatched)}")

if unmatched:
    print("\nUnmatched districts:")
    for d in sorted(unmatched):
        print(f"  {d}")
else:
    print("\n✅ ALL 261 districts now match! Hospital data can connect to population data.")

Total hospital districts: 261
Matched to population: 261
Still unmatched: 0

✅ ALL 261 districts now match! Hospital data can connect to population data.


In [12]:
# Step 2: Build the master dataset

# --- A) Prepare the all-districts-data (flatten Main/Urban/Rural into one row per district) ---

all_districts = all_sheets['all-districts-data']

# Standardize district names same way we did for hospitals
all_districts['District_upper'] = all_districts['District'].str.strip().str.upper()

# Split into Main, Urban, Rural
main_rows = all_districts[all_districts['Type'] == 'Main'].copy()
urban_rows = all_districts[all_districts['Type'] == 'Urban'].copy()
rural_rows = all_districts[all_districts['Type'] == 'Rural'].copy()

# Rename columns to be clear about what's what
main_cols = {
    'Both Sexes [Total Population]': 'Total_Population',
    'Male [Total Population]': 'Male_Total',
    'Female [Total Population]': 'Female_Total',
    'Both Sexes [Household Population]': 'Household_Total',
    'Male [Household Population]': 'Household_Male',
    'Female [Household Population]': 'Household_Female',
    'Both Sexes [Non-household Population]': 'NonHousehold_Total',
    'Male [Non-household Population]': 'NonHousehold_Male',
    'Female [Non-household Population]': 'NonHousehold_Female',
}

urban_cols = {
    'Both Sexes [Total Population]': 'Urban_Population',
    'Male [Total Population]': 'Urban_Male',
    'Female [Total Population]': 'Urban_Female',
    'Both Sexes [Household Population]': 'Urban_Household_Total',
    'Male [Household Population]': 'Urban_Household_Male',
    'Female [Household Population]': 'Urban_Household_Female',
    'Both Sexes [Non-household Population]': 'Urban_NonHousehold_Total',
    'Male [Non-household Population]': 'Urban_NonHousehold_Male',
    'Female [Non-household Population]': 'Urban_NonHousehold_Female',
}

rural_cols = {
    'Both Sexes [Total Population]': 'Rural_Population',
    'Male [Total Population]': 'Rural_Male',
    'Female [Total Population]': 'Rural_Female',
    'Both Sexes [Household Population]': 'Rural_Household_Total',
    'Male [Household Population]': 'Rural_Household_Male',
    'Female [Household Population]': 'Rural_Household_Female',
    'Both Sexes [Non-household Population]': 'Rural_NonHousehold_Total',
    'Male [Non-household Population]': 'Rural_NonHousehold_Male',
    'Female [Non-household Population]': 'Rural_NonHousehold_Female',
}

main_rows = main_rows.rename(columns=main_cols)[['Main District', 'Region'] + list(main_cols.values())]
urban_rows = urban_rows.rename(columns=urban_cols)[['Main District'] + list(urban_cols.values())]
rural_rows = rural_rows.rename(columns=rural_cols)[['Main District'] + list(rural_cols.values())]

# Merge Main + Urban + Rural into one row per district
district_demographics = main_rows.merge(urban_rows, on='Main District', how='left')
district_demographics = district_demographics.merge(rural_rows, on='Main District', how='left')

print(f"District demographics shape: {district_demographics.shape}")
print(f"Columns: {list(district_demographics.columns)}")
print(f"\nFirst 3 rows:")
district_demographics.head(3)

District demographics shape: (278, 29)
Columns: ['Main District', 'Region', 'Total_Population', 'Male_Total', 'Female_Total', 'Household_Total', 'Household_Male', 'Household_Female', 'NonHousehold_Total', 'NonHousehold_Male', 'NonHousehold_Female', 'Urban_Population', 'Urban_Male', 'Urban_Female', 'Urban_Household_Total', 'Urban_Household_Male', 'Urban_Household_Female', 'Urban_NonHousehold_Total', 'Urban_NonHousehold_Male', 'Urban_NonHousehold_Female', 'Rural_Population', 'Rural_Male', 'Rural_Female', 'Rural_Household_Total', 'Rural_Household_Male', 'Rural_Household_Female', 'Rural_NonHousehold_Total', 'Rural_NonHousehold_Male', 'Rural_NonHousehold_Female']

First 3 rows:


,Main District,Region,Total_Population,Male_Total,Female_Total,Household_Total,Household_Male,Household_Female,NonHousehold_Total,NonHousehold_Male,...,Urban_NonHousehold_Female,Rural_Population,Rural_Male,Rural_Female,Rural_Household_Total,Rural_Household_Male,Rural_Household_Female,Rural_NonHousehold_Total,Rural_NonHousehold_Male,Rural_NonHousehold_Female
0,Jomoro District,Western,126576,62649,63927,124370,61690,62680,2206,959,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Ellembelle District,Western,120893,60586,60307,117166,58930,58236,3727,1656,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Nzema East Municipal District,Western,94621,48590,46031,92933,47986,44947,1688,604,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
# Check: which districts have more than one row?
dupes = district_demographics[district_demographics['Main District'].duplicated(keep=False)]
print(f"Districts with multiple rows: {dupes['Main District'].nunique()}")
print(f"Total extra rows: {len(district_demographics) - 261}")
print(f"\nDuplicated districts:")
print(dupes[['Main District', 'Region', 'Total_Population', 'Urban_Population', 'Rural_Population']].to_string())

Districts with multiple rows: 0
Total extra rows: 17

Duplicated districts:
Empty DataFrame
Columns: [Main District, Region, Total_Population, Urban_Population, Rural_Population]
Index: []


In [14]:
# What are the 278 "Main District" values? Let's find the ones that don't look like districts
print("All Main District values:")
for i, name in enumerate(sorted(district_demographics['Main District'].unique())):
    print(f"  {i+1}. {name}")

All Main District values:
  1. Ablekuma Central Municipal District
  2. Ablekuma North Municipal District
  3. Ablekuma West Municipal District
  4. Abuakwa North District
  5. Abuakwa South Municipal District
  6. Abura Asebu Kwamankese District
  7. Accra Metropolitan Ablekuma South
  8. Accra Metropolitan Ashiedu Keteke
  9. Accra Metropolitan District
  10. Accra Metropolitan Okaikoi South
  11. Achiase District
  12. Ada East District
  13. Ada West District
  14. Adaklu District
  15. Adansi Asokwa District
  16. Adansi North District
  17. Adansi South District
  18. Adentan Municipal District
  19. Afadzato South District
  20. Afigya Kwabre North District
  21. Afigya Kwabre South District
  22. Agona East District
  23. Agona West Municipal District
  24. Agortime-Ziope District
  25. Ahafo Ano North District
  26. Ahafo Ano South East District
  27. Ahafo Ano South West District
  28. Ahanta West District
  29. Ajumaku Enyan Essiam District
  30. Akatsi North District
  31. 

In [15]:
# Remove sub-metro rows — these are subdivisions, not separate districts
# We keep only the main "District" row for each metro area

sub_metros = [
    'Accra Metropolitan Ablekuma South',
    'Accra Metropolitan Ashiedu Keteke',
    'Accra Metropolitan Okaikoi South',
    'Kumasi Metropolitan Bantama',
    'Kumasi Metropolitan Manhyia North',
    'Kumasi Metropolitan Manhyia South',
    'Kumasi Metropolitan Nhyiaeso',
    'Kumasi Metropolitan Subin',
    'Cape Coast Metropolitan Cape Coast North',
    'Cape Coast Metropolitan Cape Coast South',
    'Sekondi Takoradi Metropolitan Essikado Ketan',
    'Sekondi Takoradi Metropolitan Sekondi',
    'Sekondi Takoradi Metropolitan Takoradi',
    'Tamale Metropolitan Tamale Central',
    'Tamale Metropolitan Tamale South',
    'Tema Central District',
    'Tema East District',
]

district_demographics = district_demographics[
    ~district_demographics['Main District'].isin(sub_metros)
]

# Fix the weird double-names
district_demographics['Main District'] = district_demographics['Main District'].replace({
    'Mion District District': 'Mion District',
    'Mophor (Mpohor) Mpohor (Mpohor)': 'Mophor (Mpohor) District'
})

print(f"Rows after cleanup: {len(district_demographics)}")
print(f"Expected: 261")

if len(district_demographics) == 261:
    print("✅ Perfect! 261 districts, no extras.")
else:
    print(f"⚠️ Off by {len(district_demographics) - 261}. Need to investigate.")

Rows after cleanup: 261
Expected: 261
✅ Perfect! 261 districts, no extras.


In [16]:
# Step 3: Now merge everything together

# Create a join key for district_demographics
# Strip off "District", "Municipal District", "Metropolitan District" etc to get base name
district_demographics['join_key'] = (
    district_demographics['Main District']
    .str.upper()
    .str.replace(' MUNICIPAL DISTRICT', '', regex=False)
    .str.replace(' METROPOLITAN DISTRICT', '', regex=False)
    .str.replace(' DISTRICT', '', regex=False)
    .str.strip()
)

# The hospital data's standardized names use the population naming convention
# Let's create the same kind of join key for the population data
population_clean = population.copy()
population_clean['join_key'] = population_clean['District'].str.strip().str.upper()

# And for hospitals
hospitals_clean['join_key'] = hospitals_clean['District_standardized'].str.strip().str.upper()

# Check: do the join keys match between hospitals and district_demographics?
hosp_keys = set(hospitals_clean['join_key'].unique())
demo_keys = set(district_demographics['join_key'].unique())
pop_keys = set(population_clean['join_key'].unique())

print("Hospital keys matching demographics:", len(hosp_keys & demo_keys), "of", len(hosp_keys))
print("Hospital keys matching population:", len(hosp_keys & pop_keys), "of", len(hosp_keys))

# Show any mismatches
hosp_not_demo = hosp_keys - demo_keys
if hosp_not_demo:
    print(f"\n⚠️ {len(hosp_not_demo)} hospital keys not found in demographics:")
    for k in sorted(hosp_not_demo):
        print(f"  {k}")
else:
    print("\n✅ All hospital districts match demographics!")

hosp_not_pop = hosp_keys - pop_keys
if hosp_not_pop:
    print(f"\n⚠️ {len(hosp_not_pop)} hospital keys not found in population:")
    for k in sorted(hosp_not_pop):
        print(f"  {k}")
else:
    print("\n✅ All hospital districts match population!")

Hospital keys matching demographics: 177 of 261
Hospital keys matching population: 261 of 261

⚠️ 84 hospital keys not found in demographics:
  ABLEKUMA CENTRAL MUNICIPAL
  ABLEKUMA NORTH MUNICIPAL
  ABLEKUMA WEST MUNICIPAL
  ABUAKWA SOUTH MUNICIPAL
  ACCRA METROPOLITAN
  ADENTAN MUNICIPAL
  AGONA WEST MUNICIPAL
  AKWAPIM NORTH MUNICIPAL
  AOWIN MUNICIPAL
  ASANTE AKIM CENTRAL MUNICIPAL
  ASOKORE MAMPONG MUNICIPAL
  ASOKWA MUNICIPAL
  ASSIN CENTRAL MUNICIPAL
  ASUNAFO NORTH MUNICIPAL
  ATWIMA NWABIAGYA SOUTH MUNICIPAL
  AWUTU SENYA EAST MUNICIPAL
  AYAWASO CENTRAL MUNICIPAL
  AYAWASO EAST MUNICIPAL
  AYAWASO NORTH MUNICIPAL
  AYAWASO WEST MUNICIPAL
  BAWKU MUNICIPAL
  BEKWAI MUNICIPAL
  BEREKUM EAST MUNICIPAL
  BIRIM CENTRAL MUNICIPAL
  BOLGATANGA MUNICIPAL
  BUILSA NORTH MUNICIPAL
  CAPE COAST METROPOLITAN
  DORMAA CENTRAL MUNICIPAL
  EAST MAMPRUSI MUNICIPAL
  EFFIA KWESIMINTSIM MUNICIPAL
  EFFUTU MUNICIPAL
  EJISU JUABEN MUNICIPAL
  EJURA SEKYEDUMASE MUNICIPAL
  GA CENTRAL MUNICIPAL


In [17]:
# Fix: also strip "MUNICIPAL" and "METROPOLITAN" from the hospital join key

hospitals_clean['join_key'] = (
    hospitals_clean['District_standardized']
    .str.strip()
    .str.upper()
    .str.replace(' MUNICIPAL', '', regex=False)
    .str.replace(' METROPOLITAN', '', regex=False)
    .str.replace(' DISTRICT', '', regex=False)
    .str.strip()
)

# Also fix the population join key the same way
population_clean['join_key'] = (
    population_clean['District']
    .str.strip()
    .str.upper()
    .str.replace(' MUNICIPAL', '', regex=False)
    .str.replace(' METROPOLITAN', '', regex=False)
    .str.replace(' DISTRICT', '', regex=False)
    .str.strip()
)

# Check again
hosp_keys = set(hospitals_clean['join_key'].unique())
demo_keys = set(district_demographics['join_key'].unique())
pop_keys = set(population_clean['join_key'].unique())

print("Hospital keys matching demographics:", len(hosp_keys & demo_keys), "of", len(hosp_keys))
print("Hospital keys matching population:", len(hosp_keys & pop_keys), "of", len(hosp_keys))

hosp_not_demo = hosp_keys - demo_keys
if hosp_not_demo:
    print(f"\n⚠️ {len(hosp_not_demo)} still not matching demographics:")
    for k in sorted(hosp_not_demo):
        print(f"  {k}")
else:
    print("\n✅ All hospital districts match demographics!")

hosp_not_pop = hosp_keys - pop_keys
if hosp_not_pop:
    print(f"\n⚠️ {len(hosp_not_pop)} still not matching population:")
    for k in sorted(hosp_not_pop):
        print(f"  {k}")
else:
    print("\n✅ All hospital districts match population!")

Hospital keys matching demographics: 261 of 261
Hospital keys matching population: 261 of 261

✅ All hospital districts match demographics!

✅ All hospital districts match population!


In [18]:
# Step 4: THE ACTUAL MERGE — build the master dataset

# Merge hospital data with population data (urban/rural percentages)
master = hospitals_clean.merge(
    population_clean[['join_key', 'Main', 'Urban', 'Rural', 'Percentage of Urban', 'Percentage of Rural']],
    on='join_key',
    how='left'
)

# Rename for clarity
master = master.rename(columns={
    'Main': 'District_Population',
    'Urban': 'District_Urban_Pop',
    'Rural': 'District_Rural_Pop'
})

# Merge with district demographics (male/female, household breakdowns)
master = master.merge(
    district_demographics[['join_key', 'Male_Total', 'Female_Total',
                           'Household_Total', 'NonHousehold_Total',
                           'Urban_Male', 'Urban_Female',
                           'Rural_Male', 'Rural_Female']],
    on='join_key',
    how='left'
)

# Merge with centroids
centroids = all_sheets['district-centroids']
centroids['join_key'] = (
    centroids['District']
    .str.strip()
    .str.upper()
    .str.replace(' MUNICIPAL', '', regex=False)
    .str.replace(' METROPOLITAN', '', regex=False)
    .str.replace(' DISTRICT', '', regex=False)
    .str.strip()
)
centroids = centroids.rename(columns={'Latitude': 'Centroid_Lat', 'Longitude': 'Centroid_Lon'})

master = master.merge(
    centroids[['join_key', 'Centroid_Lat', 'Centroid_Lon']],
    on='join_key',
    how='left'
)

# Clean up — drop helper columns, keep what matters
master = master.drop(columns=['District_upper', 'join_key'])

print(f"✅ Master dataset shape: {master.shape}")
print(f"\nColumns ({len(master.columns)}):")
for col in master.columns:
    print(f"  {col}")

print(f"\nFirst 3 rows:")
master.head(3)

✅ Master dataset shape: (9978, 29)

Columns (29):
  ID
  Name
  Facility_Type
  Ownership
  Region
  District
  Sub-District
  Community
  Latitude
  Longitude
  has_emonc
  has_midwife
  has_blood_bank
  District_standardized
  District_Population
  District_Urban_Pop
  District_Rural_Pop
  Percentage of Urban
  Percentage of Rural
  Male_Total
  Female_Total
  Household_Total
  NonHousehold_Total
  Urban_Male
  Urban_Female
  Rural_Male
  Rural_Female
  Centroid_Lat
  Centroid_Lon

First 3 rows:


,ID,Name,Facility_Type,Ownership,Region,District,Sub-District,Community,Latitude,Longitude,...,Male_Total,Female_Total,Household_Total,NonHousehold_Total,Urban_Male,Urban_Female,Rural_Male,Rural_Female,Centroid_Lat,Centroid_Lon
0,4608,1 MEDICAL RECEPTION STATION(1MRS),POLYCLINIC,QUASI-GOVERNMENT,GREATER ACCRA,KPONE-KATAMANSO,GBETSILE,MICHEL CAMP,5.728982,-0.025255,...,208040,209294,416128,1206,NaN,NaN,NaN,NaN,NaN,NaN
1,564,2MRS MILITARY HOSPITAL,HOSPITAL,QUASI-GOVERNMENT,WESTERN,EFFIA-KWESIMINTSIM,APREMDO,ALREADY BARRACKS,4.914347,-1.805559,...,85864,88111,170992,2983,NaN,NaN,NaN,NaN,4.964357,-1.787798
2,9884,31ST DWM CHPS,CHPS,GOVERNMENT,ASHANTI,KUMASI,MANHYIA -ASH TOWN,ASH TOWN,6.705688,-1.621758,...,213662,230319,413561,30420,NaN,NaN,NaN,NaN,6.688186,-1.621228


In [19]:
# Quick health check on the master dataset
print("=== MASTER DATASET HEALTH CHECK ===\n")
print(f"Total facilities: {len(master)}")
print(f"Unique districts: {master['District_standardized'].nunique()}")
print(f"Unique regions: {master['Region'].nunique()}")

print(f"\n--- Missing values in key columns ---")
key_cols = ['District_Population', 'Percentage of Urban', 'Male_Total', 
            'Centroid_Lat', 'Centroid_Lon']
for col in key_cols:
    missing = master[col].isna().sum()
    print(f"  {col}: {missing} missing")

print(f"\n--- Sample: one row with all columns ---")
print(master.iloc[0].to_string())

=== MASTER DATASET HEALTH CHECK ===

Total facilities: 9978
Unique districts: 261
Unique regions: 16

--- Missing values in key columns ---
  District_Population: 0 missing
  Percentage of Urban: 0 missing
  Male_Total: 0 missing
  Centroid_Lat: 2303 missing
  Centroid_Lon: 2303 missing

--- Sample: one row with all columns ---
ID                                                    4608
Name                     1 MEDICAL RECEPTION STATION(1MRS)
Facility_Type                                   POLYCLINIC
Ownership                                 QUASI-GOVERNMENT
Region                                       GREATER ACCRA
District                                   KPONE-KATAMANSO
Sub-District                                      GBETSILE
Community                                     MICHEL CAMP 
Latitude                                          5.728982
Longitude                                        -0.025255
has_emonc                                            False
has_midwife          

In [20]:
# Which districts are missing centroids?
missing_centroids = master[master['Centroid_Lat'].isna()]['District_standardized'].unique()
print(f"Districts missing centroids: {len(missing_centroids)}")
print("\nFirst 20:")
for d in sorted(missing_centroids)[:20]:
    print(f"  {d}")

# What are the centroid join keys?
print(f"\n--- Centroid join keys sample ---")
print("Centroid keys (first 10):")
for k in sorted(centroids['join_key'].unique())[:10]:
    print(f"  {k}")

print("\nHospital keys for missing districts (first 10):")
hosp_missing_keys = master[master['Centroid_Lat'].isna()]['District_standardized'].str.upper().str.replace(' MUNICIPAL', '', regex=False).str.replace(' METROPOLITAN', '', regex=False).str.replace(' DISTRICT', '', regex=False).str.strip().unique()
for k in sorted(hosp_missing_keys)[:10]:
    print(f"  {k}")

Districts missing centroids: 56

First 20:
  ABURA ASEBU KWAMANKESE
  ADENTAN MUNICIPAL
  AFIGYA KWABRE NORTH
  AFIGYA KWABRE SOUTH
  AGORTIME-ZIOPE
  AHAFO ANO NORTH
  AHAFO ANO SOUTH EAST
  AHAFO ANO SOUTH WEST
  AJUMAKU ENYAN ESSIAM
  AKROFUOM
  AKWAPIM NORTH MUNICIPAL
  AKWAPIM SOUTH
  ASANTE AKIM CENTRAL MUNICIPAL
  ASANTE AKIM NORTH
  ASANTE AKIM SOUTH
  ASENE MANSO AKROSO
  ASIKUMA ODOBEN BRAKWA
  ASOKORE MAMPONG MUNICIPAL
  ATEBUBU AMANTIN
  ATWIMA KWANWOMA

--- Centroid join keys sample ---
Centroid keys (first 10):
  ABLEKUMA CENTRAL
  ABLEKUMA NORTH
  ABLEKUMA WEST
  ABUAKWA NORTH
  ABUAKWA SOUTH
  ABURA-ASEBU-KWAMANKESE
  ACCRA
  ACHIASE
  ADA EAST
  ADA WEST

Hospital keys for missing districts (first 10):
  ABURA ASEBU KWAMANKESE
  ADENTAN
  AFIGYA KWABRE NORTH
  AFIGYA KWABRE SOUTH
  AGORTIME-ZIOPE
  AHAFO ANO NORTH
  AHAFO ANO SOUTH EAST
  AHAFO ANO SOUTH WEST
  AJUMAKU ENYAN ESSIAM
  AKROFUOM


In [21]:
# Create a simplified key that removes ALL hyphens, extra spaces, 
# and common suffixes — so we match on the core name only

def make_simple_key(name):
    return (
        str(name)
        .upper()
        .replace('-', ' ')
        .replace('/', ' ')
        .replace('(', '')
        .replace(')', '')
        .replace(' MUNICIPAL', '')
        .replace(' METROPOLITAN', '')
        .replace(' DISTRICT', '')
        .replace('  ', ' ')
        .strip()
    )

# Apply to all datasets
master['simple_key'] = master['District_standardized'].apply(make_simple_key)
centroids['simple_key'] = centroids['District'].apply(make_simple_key)
district_demographics['simple_key'] = district_demographics['Main District'].apply(make_simple_key)

# Check centroid matching with simple key
master_keys = set(master['simple_key'].unique())
centroid_keys = set(centroids['simple_key'].unique())
demo_keys = set(district_demographics['simple_key'].unique())

print("Centroids match:", len(master_keys & centroid_keys), "of", len(master_keys))
still_missing_cent = master_keys - centroid_keys
if still_missing_cent:
    print(f"\nStill missing centroids ({len(still_missing_cent)}):")
    for k in sorted(still_missing_cent):
        # Find what the centroid version looks like
        print(f"  Hospital: {k}")
else:
    print("✅ All centroid matches found!")

print(f"\nDemographics match:", len(master_keys & demo_keys), "of", len(master_keys))
still_missing_demo = master_keys - demo_keys
if still_missing_demo:
    print(f"\nStill missing demographics ({len(still_missing_demo)}):")
    for k in sorted(still_missing_demo):
        print(f"  Hospital: {k}")
else:
    print("✅ All demographics matches found!")

Centroids match: 234 of 261

Still missing centroids (27):
  Hospital: ADENTAN
  Hospital: AGORTIME ZIOPE
  Hospital: AJUMAKU ENYAN ESSIAM
  Hospital: AKROFUOM
  Hospital: AKWAPIM NORTH
  Hospital: AKWAPIM SOUTH
  Hospital: BOLGATANGA EAST
  Hospital: BOSOMTWI
  Hospital: BUNKPURUGU NYANKPANDURI
  Hospital: DAFFIAMA BUSSIE
  Hospital: DORMAA CENTRAL
  Hospital: EJISU JUABEN
  Hospital: KASSENA NANKANA EAST
  Hospital: KASSENA NANKANA WEST
  Hospital: LA DADEKOTOPON
  Hospital: LA NKWANTANAN MADINA
  Hospital: MAMPONG
  Hospital: MFANTSIMAN
  Hospital: MOPHOR MPOHOR
  Hospital: OKAI KOI NORTH
  Hospital: SAGNARIGU
  Hospital: SEFWI BIBIANI AHWIASO BEKWAI
  Hospital: SEKYERE AFRAM PLAINS
  Hospital: TATALE
  Hospital: TWIFO ATI MORKWA
  Hospital: TWIFO HEMAN LOWER DENKYIRA
  Hospital: UPPER MANYA KROBO

Demographics match: 261 of 261
✅ All demographics matches found!


In [22]:
# Re-merge centroids using simple_key
centroid_merge = centroids[['simple_key', 'Centroid_Lat', 'Centroid_Lon']].drop_duplicates(subset='simple_key')

master = master.drop(columns=['Centroid_Lat', 'Centroid_Lon'])
master = master.merge(centroid_merge, on='simple_key', how='left')

# Re-merge the urban/rural gender breakdown using simple_key
demo_merge = district_demographics[['simple_key', 'Urban_Male', 'Urban_Female', 
                                     'Rural_Male', 'Rural_Female']].drop_duplicates(subset='simple_key')

master = master.drop(columns=['Urban_Male', 'Urban_Female', 'Rural_Male', 'Rural_Female'])
master = master.merge(demo_merge, on='simple_key', how='left')

# Drop helper column
master = master.drop(columns=['simple_key'])

# Final health check
print("=== FINAL HEALTH CHECK ===\n")
print(f"Total facilities: {len(master)}")
print(f"Unique districts: {master['District_standardized'].nunique()}")

print(f"\n--- Missing values in ALL columns ---")
for col in master.columns:
    missing = master[col].isna().sum()
    if missing > 0:
        print(f"  {col}: {missing} missing")

if master[['Centroid_Lat', 'Centroid_Lon', 'Urban_Male', 'Urban_Female']].isna().sum().sum() == 0:
    print("\n✅ Master dataset is complete! Phase 1 DONE.")
else:
    print("\n⚠️ Still have some gaps — let's investigate.")

=== FINAL HEALTH CHECK ===

Total facilities: 9978
Unique districts: 261

--- Missing values in ALL columns ---
  District_Rural_Pop: 550 missing
  Centroid_Lat: 1041 missing
  Centroid_Lon: 1041 missing
  Urban_Male: 9978 missing
  Urban_Female: 9978 missing
  Rural_Male: 9978 missing
  Rural_Female: 9978 missing

⚠️ Still have some gaps — let's investigate.


In [23]:
# Check: did the demographics simple keys actually get created properly?
print("Demographics simple_key sample (first 10):")
for k in sorted(district_demographics['simple_key'].unique())[:10]:
    print(f"  '{k}'")

print("\nMaster simple_key sample (first 10):")
for k in sorted(master['District_standardized'].apply(
    lambda x: str(x).upper().replace('-', ' ').replace('/', ' ')
    .replace('(', '').replace(')', '')
    .replace(' MUNICIPAL', '').replace(' METROPOLITAN', '')
    .replace(' DISTRICT', '').replace('  ', ' ').strip()
).unique())[:10]:
    print(f"  '{k}'")

# Check if simple_key column still exists on master after we dropped it
print(f"\n'simple_key' in master columns: {'simple_key' in master.columns}")

# Check what happened with the demo merge
print(f"\nUrban_Male null count: {master['Urban_Male'].isna().sum()}")
print(f"Urban_Male dtype: {master['Urban_Male'].dtype}")

# Let's check if it's a column content issue
print(f"\nDistrict demographics Urban_Male sample:")
print(district_demographics[['Main District', 'Urban_Male']].head(10).to_string())

Demographics simple_key sample (first 10):
  'ABLEKUMA CENTRAL'
  'ABLEKUMA NORTH'
  'ABLEKUMA WEST'
  'ABUAKWA NORTH'
  'ABUAKWA SOUTH'
  'ABURA ASEBU KWAMANKESE'
  'ACCRA'
  'ACHIASE'
  'ADA EAST'
  'ADA WEST'

Master simple_key sample (first 10):
  'ABLEKUMA CENTRAL'
  'ABLEKUMA NORTH'
  'ABLEKUMA WEST'
  'ABUAKWA NORTH'
  'ABUAKWA SOUTH'
  'ABURA ASEBU KWAMANKESE'
  'ACCRA'
  'ACHIASE'
  'ADA EAST'
  'ADA WEST'

'simple_key' in master columns: False

Urban_Male null count: 9978
Urban_Male dtype: object

District demographics Urban_Male sample:
                             Main District Urban_Male
0                          Jomoro District        NaN
1                      Ellembelle District        NaN
2            Nzema East Municipal District        NaN
3                     Ahanta West District        NaN
4    Effia Kwesimintsim Municipal District        NaN
5   Sekondi Takoradi Metropolitan District        NaN
9                           Shama District        NaN
10            

In [24]:
# Check: what does the raw all-districts-data look like for urban/rural?
all_districts = all_sheets['all-districts-data']

# Look at one district to understand the structure
sample_district = all_districts[all_districts['District'].str.contains('Jomoro', na=False)]
print("Jomoro rows:")
print(sample_district[['Type', 'District', 'Main District', 'Both Sexes [Total Population]', 
                        'Male [Total Population]', 'Female [Total Population]']].to_string())

print(f"\n\nAll unique 'Type' values: {all_districts['Type'].unique()}")
print(f"\nRows per Type:")
print(all_districts['Type'].value_counts())

Jomoro rows:
    Type District    Main District Both Sexes [Total Population] Male [Total Population] Female [Total Population]
0   Main   Jomoro  Jomoro District                        126576                   62649                     63927
1  Urban   Jomoro     Jomoro Urban                         38072                   18481                     19591
2  Rural   Jomoro     Jomoro Rural                         88504                   44168                     44336


All unique 'Type' values: ['Main' 'Urban' 'Rural']

Rows per Type:
Type
Main     278
Urban    278
Rural    278
Name: count, dtype: int64


In [25]:
# Redo the flatten using 'District' column as the join key instead of 'Main District'

all_districts = all_sheets['all-districts-data']

main_rows = all_districts[all_districts['Type'] == 'Main'].copy()
urban_rows = all_districts[all_districts['Type'] == 'Urban'].copy()
rural_rows = all_districts[all_districts['Type'] == 'Rural'].copy()

# Rename columns
main_cols = {
    'Both Sexes [Total Population]': 'Total_Population',
    'Male [Total Population]': 'Male_Total',
    'Female [Total Population]': 'Female_Total',
    'Both Sexes [Household Population]': 'Household_Total',
    'Male [Household Population]': 'Household_Male',
    'Female [Household Population]': 'Household_Female',
    'Both Sexes [Non-household Population]': 'NonHousehold_Total',
    'Male [Non-household Population]': 'NonHousehold_Male',
    'Female [Non-household Population]': 'NonHousehold_Female',
}

urban_cols = {
    'Both Sexes [Total Population]': 'Urban_Population',
    'Male [Total Population]': 'Urban_Male',
    'Female [Total Population]': 'Urban_Female',
    'Both Sexes [Household Population]': 'Urban_Household_Total',
    'Male [Household Population]': 'Urban_Household_Male',
    'Female [Household Population]': 'Urban_Household_Female',
    'Both Sexes [Non-household Population]': 'Urban_NonHousehold_Total',
    'Male [Non-household Population]': 'Urban_NonHousehold_Male',
    'Female [Non-household Population]': 'Urban_NonHousehold_Female',
}

rural_cols = {
    'Both Sexes [Total Population]': 'Rural_Population',
    'Male [Total Population]': 'Rural_Male',
    'Female [Total Population]': 'Rural_Female',
    'Both Sexes [Household Population]': 'Rural_Household_Total',
    'Male [Household Population]': 'Rural_Household_Male',
    'Female [Household Population]': 'Rural_Household_Female',
    'Both Sexes [Non-household Population]': 'Rural_NonHousehold_Total',
    'Male [Non-household Population]': 'Rural_NonHousehold_Male',
    'Female [Non-household Population]': 'Rural_NonHousehold_Female',
}

# Use 'District' as the join key — it's consistent across all three types
main_renamed = main_rows.rename(columns=main_cols)[['District', 'Main District', 'Region'] + list(main_cols.values())]
urban_renamed = urban_rows.rename(columns=urban_cols)[['District'] + list(urban_cols.values())]
rural_renamed = rural_rows.rename(columns=rural_cols)[['District'] + list(rural_cols.values())]

# Merge on District
district_demographics = main_renamed.merge(urban_renamed, on='District', how='left')
district_demographics = district_demographics.merge(rural_renamed, on='District', how='left')

# Remove sub-metro rows
sub_metros = [
    'Accra Metropolitan Ablekuma South',
    'Accra Metropolitan Ashiedu Keteke',
    'Accra Metropolitan Okaikoi South',
    'Kumasi Metropolitan Bantama',
    'Kumasi Metropolitan Manhyia North',
    'Kumasi Metropolitan Manhyia South',
    'Kumasi Metropolitan Nhyiaeso',
    'Kumasi Metropolitan Subin',
    'Cape Coast Metropolitan Cape Coast North',
    'Cape Coast Metropolitan Cape Coast South',
    'Sekondi Takoradi Metropolitan Essikado Ketan',
    'Sekondi Takoradi Metropolitan Sekondi',
    'Sekondi Takoradi Metropolitan Takoradi',
    'Tamale Metropolitan Tamale Central',
    'Tamale Metropolitan Tamale South',
    'Tema Central District',
    'Tema East District',
]
district_demographics = district_demographics[~district_demographics['Main District'].isin(sub_metros)]

# Fix weird names
district_demographics['Main District'] = district_demographics['Main District'].replace({
    'Mion District District': 'Mion District',
    'Mophor (Mpohor) Mpohor (Mpohor)': 'Mophor (Mpohor) District'
})

print(f"Shape: {district_demographics.shape}")
print(f"\nSample — Jomoro:")
jomoro = district_demographics[district_demographics['District'] == 'Jomoro']
print(f"  Urban_Male: {jomoro['Urban_Male'].values}")
print(f"  Rural_Female: {jomoro['Rural_Female'].values}")
print(f"  Total_Population: {jomoro['Total_Population'].values}")

Shape: (261, 30)

Sample — Jomoro:
  Urban_Male: [18481]
  Rural_Female: [44336]
  Total_Population: [126576]


In [26]:
# Step 5: REBUILD MASTER DATASET — clean from scratch

# Create simple_key function
def make_simple_key(name):
    return (
        str(name)
        .upper()
        .replace('-', ' ')
        .replace('/', ' ')
        .replace('(', '')
        .replace(')', '')
        .replace(' MUNICIPAL', '')
        .replace(' METROPOLITAN', '')
        .replace(' DISTRICT', '')
        .replace('  ', ' ')
        .strip()
    )

# Add simple_key to all datasets
hospitals_clean['simple_key'] = hospitals_clean['District_standardized'].apply(make_simple_key)

district_demographics['simple_key'] = district_demographics['Main District'].apply(make_simple_key)

centroids = all_sheets['district-centroids'].copy()
centroids['simple_key'] = centroids['District'].apply(make_simple_key)
centroids = centroids.rename(columns={'Latitude': 'Centroid_Lat', 'Longitude': 'Centroid_Lon'})

population_clean = population.copy()
population_clean['simple_key'] = population_clean['District'].apply(make_simple_key)

# Start with hospitals
master = hospitals_clean[['ID', 'Name', 'Facility_Type', 'Ownership', 'Region', 'District',
                           'Sub-District', 'Community', 'Latitude', 'Longitude',
                           'has_emonc', 'has_midwife', 'has_blood_bank',
                           'District_standardized', 'simple_key']].copy()

# Merge population (urban/rural percentages)
master = master.merge(
    population_clean[['simple_key', 'Main', 'Urban', 'Rural', 'Percentage of Urban', 'Percentage of Rural']].drop_duplicates(subset='simple_key'),
    on='simple_key', how='left'
).rename(columns={'Main': 'District_Population', 'Urban': 'District_Urban_Pop', 'Rural': 'District_Rural_Pop'})

# Merge demographics (male/female, household, urban/rural breakdowns)
demo_cols = ['simple_key', 'Male_Total', 'Female_Total', 
             'Household_Total', 'NonHousehold_Total',
             'Urban_Male', 'Urban_Female', 'Rural_Male', 'Rural_Female',
             'Urban_Population', 'Rural_Population']
master = master.merge(
    district_demographics[demo_cols].drop_duplicates(subset='simple_key'),
    on='simple_key', how='left'
)

# Merge centroids
master = master.merge(
    centroids[['simple_key', 'Centroid_Lat', 'Centroid_Lon']].drop_duplicates(subset='simple_key'),
    on='simple_key', how='left'
)

# Drop helper column
master = master.drop(columns=['simple_key'])

# FINAL HEALTH CHECK
print("=== FINAL HEALTH CHECK ===\n")
print(f"Total facilities: {len(master)}")
print(f"Unique districts: {master['District_standardized'].nunique()}")
print(f"Unique regions: {master['Region'].nunique()}")
print(f"Columns: {len(master.columns)}")

print(f"\n--- Missing values ---")
any_missing = False
for col in master.columns:
    missing = master[col].isna().sum()
    if missing > 0:
        print(f"  {col}: {missing} missing ({round(missing/len(master)*100, 1)}%)")
        any_missing = True

if not any_missing:
    print("  ZERO missing values anywhere!")
    print("\n✅ PHASE 1 COMPLETE. Master dataset is ready!")

=== FINAL HEALTH CHECK ===

Total facilities: 9978
Unique districts: 261
Unique regions: 16
Columns: 31

--- Missing values ---
  District_Rural_Pop: 550 missing (5.5%)
  Rural_Male: 807 missing (8.1%)
  Rural_Female: 807 missing (8.1%)
  Rural_Population: 807 missing (8.1%)
  Centroid_Lat: 1041 missing (10.4%)
  Centroid_Lon: 1041 missing (10.4%)


In [27]:
# Issue 1: Are the missing rural values legitimate (fully urban districts)?
missing_rural = master[master['District_Rural_Pop'].isna()]['District_standardized'].unique()
print(f"Districts with missing Rural Pop: {len(missing_rural)}")
print("\nThese districts and their urban percentage:")
for d in sorted(missing_rural)[:15]:
    row = master[master['District_standardized'] == d].iloc[0]
    print(f"  {d}: {row['Percentage of Urban']*100:.1f}% urban")

# Issue 2: Which districts are missing centroids?
missing_cent = master[master['Centroid_Lat'].isna()]['District_standardized'].unique()
print(f"\n\nDistricts missing centroids: {len(missing_cent)}")

# Compare simple keys
master_temp = master[master['Centroid_Lat'].isna()].copy()
master_keys_missing = set(master_temp['District_standardized'].apply(make_simple_key).unique())

centroid_keys_all = set(centroids['simple_key'].unique())
centroid_keys_unused = centroid_keys_all - set(master[master['Centroid_Lat'].notna()]['District_standardized'].apply(make_simple_key).unique())

print(f"\nMissing from master (first 15):")
for k in sorted(master_keys_missing)[:15]:
    print(f"  {k}")

print(f"\nUnused centroid keys (first 15):")
for k in sorted(centroid_keys_unused)[:15]:
    print(f"  {k}")

Districts with missing Rural Pop: 11

These districts and their urban percentage:
  ADENTAN MUNICIPAL: 100.0% urban
  ASHAIMAN: 100.0% urban
  GA CENTRAL MUNICIPAL: 100.0% urban
  GA NORTH: 100.0% urban
  KORLE KLOTTEY: 100.0% urban
  KROWOR: 100.0% urban
  LA DADEKOTOPON MUNICIPAL: 100.0% urban
  LEDZOKUKU: 100.0% urban
  OKAI KOI NORTH: 100.0% urban
  TEMA WEST: 100.0% urban
  WEIJA GBAWE MUNICIPAL: 100.0% urban


Districts missing centroids: 27

Missing from master (first 15):
  ADENTAN
  AGORTIME ZIOPE
  AJUMAKU ENYAN ESSIAM
  AKROFUOM
  AKWAPIM NORTH
  AKWAPIM SOUTH
  BOLGATANGA EAST
  BOSOMTWI
  BUNKPURUGU NYANKPANDURI
  DAFFIAMA BUSSIE
  DORMAA CENTRAL
  EJISU JUABEN
  KASSENA NANKANA EAST
  KASSENA NANKANA WEST
  LA DADEKOTOPON

Unused centroid keys (first 15):
  ADANSI AKROFUOM
  ADENTA
  AGOTIME ZIOPE
  AJUMAKO ENYAN ESSIAM
  AKWAPEM NORTH
  AKWAPEM SOUTH
  ASANTE MAMPONG
  BIBIANI ANHWIASO BEKWAI
  BOLGA EAST
  BOSOMTWE
  BUNKPURUGU NAKPANDURI
  DAFFIAMA BUSSIE ISSA
  DORMAA

In [28]:
# Show all 27 side by side so we can map them
print(f"{'MASTER KEY':<40} {'CLOSEST CENTROID KEY'}")
print("="*80)

from fuzzywuzzy import fuzz, process

master_missing = sorted(master_keys_missing)
centroid_unused = sorted(centroid_keys_unused)

for mk in master_missing:
    best_match, score = process.extractOne(mk, centroid_unused, scorer=fuzz.token_sort_ratio)
    print(f"{mk:<40} {best_match:<30} (score: {score})")

MASTER KEY                               CLOSEST CENTROID KEY
ADENTAN                                  ADENTA                         (score: 92)
AGORTIME ZIOPE                           AGOTIME ZIOPE                  (score: 96)
AJUMAKU ENYAN ESSIAM                     AJUMAKO ENYAN ESSIAM           (score: 95)
AKROFUOM                                 ADANSI AKROFUOM                (score: 70)
AKWAPIM NORTH                            AKWAPEM NORTH                  (score: 92)
AKWAPIM SOUTH                            AKWAPEM SOUTH                  (score: 92)
BOLGATANGA EAST                          BOLGA EAST                     (score: 80)
BOSOMTWI                                 BOSOMTWE                       (score: 88)
BUNKPURUGU NYANKPANDURI                  BUNKPURUGU NAKPANDURI          (score: 95)
DAFFIAMA BUSSIE                          DAFFIAMA BUSSIE ISSA           (score: 86)
DORMAA CENTRAL                           DORMAA                         (score: 60)
EJISU JUABEN  

In [29]:
# Manual centroid mapping: master simple_key -> centroid simple_key
centroid_fix = {
    'ADENTAN': 'ADENTA',
    'AGORTIME ZIOPE': 'AGOTIME ZIOPE',
    'AJUMAKU ENYAN ESSIAM': 'AJUMAKO ENYAN ESSIAM',
    'AKROFUOM': 'ADANSI AKROFUOM',
    'AKWAPIM NORTH': 'AKWAPEM NORTH',
    'AKWAPIM SOUTH': 'AKWAPEM SOUTH',
    'BOLGATANGA EAST': 'BOLGA EAST',
    'BOSOMTWI': 'BOSOMTWE',
    'BUNKPURUGU NYANKPANDURI': 'BUNKPURUGU NAKPANDURI',
    'DAFFIAMA BUSSIE': 'DAFFIAMA BUSSIE ISSA',
    'DORMAA CENTRAL': 'DORMAA',
    'EJISU JUABEN': 'EJISU',
    'KASSENA NANKANA EAST': 'KASENA NANKANA EAST',
    'KASSENA NANKANA WEST': 'KASENA NANKANA WEST',
    'LA DADEKOTOPON': 'LA DADE KOTOPON',
    'LA NKWANTANAN MADINA': 'LA NKWANTANANG MADINA',
    'MAMPONG': 'ASANTE MAMPONG',
    'MFANTSIMAN': 'MFANTSEMAN',
    'MOPHOR MPOHOR': 'MPOHOR',
    'OKAI KOI NORTH': 'OKAIKWEI NORTH',
    'SAGNARIGU': 'SAGNERIGU',
    'SEFWI BIBIANI AHWIASO BEKWAI': 'BIBIANI ANHWIASO BEKWAI',
    'SEKYERE AFRAM PLAINS': 'SEKYERE AFRAM PLAINS NORTH',
    'TATALE': 'TATALE SANGULI',
    'TWIFO ATI MORKWA': 'TWIFO ATTI MORKWA',
    'TWIFO HEMAN LOWER DENKYIRA': 'TWIFO HEMANG LOWER DENKYIRA',
    'UPPER MANYA KROBO': 'UPPER MANYA',
}

# Apply: for each hospital missing a centroid, look up via the fix mapping
master['simple_key'] = master['District_standardized'].apply(make_simple_key)

# Map the simple_key to centroid's simple_key
master['centroid_key'] = master['simple_key'].map(lambda x: centroid_fix.get(x, x))

# Build centroid lookup
centroid_lookup = centroids.set_index('simple_key')[['Centroid_Lat', 'Centroid_Lon']].to_dict('index')

# Fill in missing centroids
for idx, row in master[master['Centroid_Lat'].isna()].iterrows():
    ck = row['centroid_key']
    if ck in centroid_lookup:
        master.at[idx, 'Centroid_Lat'] = centroid_lookup[ck]['Centroid_Lat']
        master.at[idx, 'Centroid_Lon'] = centroid_lookup[ck]['Centroid_Lon']

# Drop helper columns
master = master.drop(columns=['simple_key', 'centroid_key'])

# FINAL FINAL health check
print("=== FINAL HEALTH CHECK ===\n")
print(f"Total facilities: {len(master)}")
print(f"Unique districts: {master['District_standardized'].nunique()}")

print(f"\n--- Missing values ---")
any_real_issue = False
for col in master.columns:
    missing = master[col].isna().sum()
    if missing > 0:
        print(f"  {col}: {missing} missing ({round(missing/len(master)*100, 1)}%)")
        if col not in ['District_Rural_Pop', 'Rural_Male', 'Rural_Female', 'Rural_Population']:
            any_real_issue = True

if not any_real_issue:
    print("\n✅ PHASE 1 COMPLETE!")
    print("Only missing values are rural data for 100% urban districts — which is correct.")
    print(f"\nMaster dataset: {master.shape[0]} facilities x {master.shape[1]} columns")
    print("Ready for Phase 2!")

=== FINAL HEALTH CHECK ===

Total facilities: 9978
Unique districts: 261

--- Missing values ---
  District_Rural_Pop: 550 missing (5.5%)
  Rural_Male: 807 missing (8.1%)
  Rural_Female: 807 missing (8.1%)
  Rural_Population: 807 missing (8.1%)

✅ PHASE 1 COMPLETE!
Only missing values are rural data for 100% urban districts — which is correct.

Master dataset: 9978 facilities x 31 columns
Ready for Phase 2!


In [30]:
print(master.shape)

(9978, 31)


In [31]:
master.head(10)

,ID,Name,Facility_Type,Ownership,Region,District,Sub-District,Community,Latitude,Longitude,...,Household_Total,NonHousehold_Total,Urban_Male,Urban_Female,Rural_Male,Rural_Female,Urban_Population,Rural_Population,Centroid_Lat,Centroid_Lon
0,4608,1 MEDICAL RECEPTION STATION(1MRS),POLYCLINIC,QUASI-GOVERNMENT,GREATER ACCRA,KPONE-KATAMANSO,GBETSILE,MICHEL CAMP,5.728982,-0.025255,...,416128,1206,196707,198175,11333,11119,394882,22452,5.756751,-0.059359
1,564,2MRS MILITARY HOSPITAL,HOSPITAL,QUASI-GOVERNMENT,WESTERN,EFFIA-KWESIMINTSIM,APREMDO,ALREADY BARRACKS,4.914347,-1.805559,...,170992,2983,85864,88111,-,-,173975,-,4.964357,-1.787798
2,9884,31ST DWM CHPS,CHPS,GOVERNMENT,ASHANTI,KUMASI,MANHYIA -ASH TOWN,ASH TOWN,6.705688,-1.621758,...,413561,30420,213662,230319,-,-,443981,-,6.688186,-1.621228
3,3955,37 MILITARY HOSPITAL,HOSPITAL,QUASI-GOVERNMENT,GREATER ACCRA,AYAWASO EAST,KANDA,<NULL>,5.588470,-0.183260,...,52508,496,25438,27566,NaN,NaN,53004,NaN,5.590851,-0.191775
4,4865,3E MEDICAL CENTRE,CLINIC,PRIVATE,GREATER ACCRA,GA SOUTH,BORTIANOR,BORTIANOR REDTOP,5.531571,-0.351723,...,349171,950,131068,135653,41424,41976,266721,83400,5.674765,-0.436827
5,4098,3M&C GHANA LIMITED,HEALTH CENTRE,PRIVATE,GREATER ACCRA,AYAWASO WEST,LEGON,LEGON,5.643030,-0.153326,...,60952,14351,38614,36689,NaN,NaN,75303,NaN,5.633869,-0.173598
6,7900,3MRS SUNYANI,HOSPITAL,QUASI-GOVERNMENT,BONO,SUNYANI MUNICIPAL,NEW DORMAA,ASUAKWAH,7.341959,-2.294543,...,185031,8564,77088,79255,19270,17982,156343,37252,7.234701,-2.364255
7,10078,3WAY FAMILY CARE CLINIC(CLOSED),CLINIC,PRIVATE,AHAFO,ASUTIFI SOUTH,ACHERENSUA,KROFROM,6.979922,-2.289259,...,66692,1702,16515,16721,18417,16741,33236,35158,6.837113,-2.390411
8,9130,4MRS CLINIC,CLINIC,QUASI-GOVERNMENT,ASHANTI,KUMASI,SUBIN NORTH,4BM BARRACKS,6.694661,-1.629465,...,413561,30420,213662,230319,-,-,443981,-,6.688186,-1.621228
9,3780,64 BENCH CHPS,CHPS,GOVERNMENT,NORTHERN,TAMALE,TAMALE CENTRAL,CHANGLI,9.395928,-0.838161,...,365510,9234,185051,189693,-,-,374744,-,9.373733,-0.743034


In [32]:
pd.set_option('display.max_columns', None)
master.head(10)

,ID,Name,Facility_Type,Ownership,Region,District,Sub-District,Community,Latitude,Longitude,has_emonc,has_midwife,has_blood_bank,District_standardized,District_Population,District_Urban_Pop,District_Rural_Pop,Percentage of Urban,Percentage of Rural,Male_Total,Female_Total,Household_Total,NonHousehold_Total,Urban_Male,Urban_Female,Rural_Male,Rural_Female,Urban_Population,Rural_Population,Centroid_Lat,Centroid_Lon
0,4608,1 MEDICAL RECEPTION STATION(1MRS),POLYCLINIC,QUASI-GOVERNMENT,GREATER ACCRA,KPONE-KATAMANSO,GBETSILE,MICHEL CAMP,5.728982,-0.025255,False,False,False,KPONE KATAMANSO,417334,394882,22452,0.946201,0.053799,208040,209294,416128,1206,196707,198175,11333,11119,394882,22452,5.756751,-0.059359
1,564,2MRS MILITARY HOSPITAL,HOSPITAL,QUASI-GOVERNMENT,WESTERN,EFFIA-KWESIMINTSIM,APREMDO,ALREADY BARRACKS,4.914347,-1.805559,False,False,False,EFFIA KWESIMINTSIM MUNICIPAL,173975,173975,-,1.000000,0.000000,85864,88111,170992,2983,85864,88111,-,-,173975,-,4.964357,-1.787798
2,9884,31ST DWM CHPS,CHPS,GOVERNMENT,ASHANTI,KUMASI,MANHYIA -ASH TOWN,ASH TOWN,6.705688,-1.621758,False,False,False,KUMASI METROPOLITAN,443981,443981,-,1.000000,0.000000,213662,230319,413561,30420,213662,230319,-,-,443981,-,6.688186,-1.621228
3,3955,37 MILITARY HOSPITAL,HOSPITAL,QUASI-GOVERNMENT,GREATER ACCRA,AYAWASO EAST,KANDA,<NULL>,5.588470,-0.183260,False,False,False,AYAWASO EAST MUNICIPAL,53004,53004,0,1.000000,0.000000,25438,27566,52508,496,25438,27566,NaN,NaN,53004,NaN,5.590851,-0.191775
4,4865,3E MEDICAL CENTRE,CLINIC,PRIVATE,GREATER ACCRA,GA SOUTH,BORTIANOR,BORTIANOR REDTOP,5.531571,-0.351723,False,False,False,GA SOUTH,350121,266721,83400,0.761797,0.238203,172492,177629,349171,950,131068,135653,41424,41976,266721,83400,5.674765,-0.436827
5,4098,3M&C GHANA LIMITED,HEALTH CENTRE,PRIVATE,GREATER ACCRA,AYAWASO WEST,LEGON,LEGON,5.643030,-0.153326,False,False,False,AYAWASO WEST MUNICIPAL,75303,75303,0,1.000000,0.000000,38614,36689,60952,14351,38614,36689,NaN,NaN,75303,NaN,5.633869,-0.173598
6,7900,3MRS SUNYANI,HOSPITAL,QUASI-GOVERNMENT,BONO,SUNYANI MUNICIPAL,NEW DORMAA,ASUAKWAH,7.341959,-2.294543,False,False,False,SUNYANI MUNICIPAL,193595,156343,37252,0.807578,0.192422,96358,97237,185031,8564,77088,79255,19270,17982,156343,37252,7.234701,-2.364255
7,10078,3WAY FAMILY CARE CLINIC(CLOSED),CLINIC,PRIVATE,AHAFO,ASUTIFI SOUTH,ACHERENSUA,KROFROM,6.979922,-2.289259,False,False,False,ASUTIFI SOUTH,68394,33236,35158,0.485949,0.514051,34932,33462,66692,1702,16515,16721,18417,16741,33236,35158,6.837113,-2.390411
8,9130,4MRS CLINIC,CLINIC,QUASI-GOVERNMENT,ASHANTI,KUMASI,SUBIN NORTH,4BM BARRACKS,6.694661,-1.629465,False,False,False,KUMASI METROPOLITAN,443981,443981,-,1.000000,0.000000,213662,230319,413561,30420,213662,230319,-,-,443981,-,6.688186,-1.621228
9,3780,64 BENCH CHPS,CHPS,GOVERNMENT,NORTHERN,TAMALE,TAMALE CENTRAL,CHANGLI,9.395928,-0.838161,False,False,False,TAMALE METROPOLITAN,374744,374744,-,1.000000,0.000000,185051,189693,365510,9234,185051,189693,-,-,374744,-,9.373733,-0.743034


In [33]:
# Add the distance from centroid to master
distance_data = all_sheets['hospital-to-centroids-km'][['ID', 'distance_from_centroid_km']]

master = master.merge(distance_data, on='ID', how='left')

print(f"Master shape now: {master.shape}")
print(f"Missing distance values: {master['distance_from_centroid_km'].isna().sum()}")
print(f"\nDistance stats:")
print(master['distance_from_centroid_km'].describe())

Master shape now: (9978, 32)
Missing distance values: 1

Distance stats:
count    9977.000000
mean       11.609202
std        10.193743
min         0.048000
25%         4.901000
50%         9.384000
75%        15.517000
max       301.761000
Name: distance_from_centroid_km, dtype: float64


In [34]:
print(master[master['distance_from_centroid_km'].isna()][['Name', 'District', 'Region', 'Latitude', 'Longitude']])

                   Name      District         Region  Latitude  Longitude
3  37 MILITARY HOSPITAL  AYAWASO EAST  GREATER ACCRA   5.58847   -0.18326


In [35]:
# That's the 37 Military Hospital — one of the biggest hospitals in Ghana! It has coordinates but no distance calculated. Probably because it was the 
# one facility that had missing coordinates in the original hospital-to-centroids-km sheet before I cleaned it.
# We can calculate its distance ourselves since we have both its coordinates and the centroid:

from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

# Get the centroid for Ayawaso East
row = master[master['distance_from_centroid_km'].isna()].iloc[0]
dist = haversine(row['Latitude'], row['Longitude'], row['Centroid_Lat'], row['Centroid_Lon'])

master.loc[master['distance_from_centroid_km'].isna(), 'distance_from_centroid_km'] = round(dist, 3)

print(f"37 Military Hospital distance from Ayawaso East centroid: {round(dist, 3)} km")
print(f"Missing distances now: {master['distance_from_centroid_km'].isna().sum()}")

37 Military Hospital distance from Ayawaso East centroid: 0.979 km
Missing distances now: 0


In [36]:
# Min distance
min_row = master.loc[master['distance_from_centroid_km'].idxmin()]
print("=== CLOSEST FACILITY TO CENTROID ===")
print(f"  Name: {min_row['Name']}")
print(f"  District: {min_row['District']}")
print(f"  Region: {min_row['Region']}")
print(f"  Distance: {min_row['distance_from_centroid_km']} km")

# Max distance
max_row = master.loc[master['distance_from_centroid_km'].idxmax()]
print(f"\n=== FARTHEST FACILITY FROM CENTROID ===")
print(f"  Name: {max_row['Name']}")
print(f"  District: {max_row['District']}")
print(f"  Region: {max_row['Region']}")
print(f"  Distance: {max_row['distance_from_centroid_km']} km")

=== CLOSEST FACILITY TO CENTROID ===
  Name: CWM CHURCH AREA CHPS
  District: ASHAIMAN
  Region: GREATER ACCRA
  Distance: 0.048 km

=== FARTHEST FACILITY FROM CENTROID ===
  Name: ABUADI CHPS
  District: ADAKLU
  Region: VOLTA
  Distance: 301.761 km


In [37]:
# Check facilities with unusually large distances (say over 50km)
far_facilities = master[master['distance_from_centroid_km'] > 50].sort_values('distance_from_centroid_km', ascending=False)

print(f"Facilities more than 50km from centroid: {len(far_facilities)}")
print(f"\n{'Name':<40} {'District':<25} {'Region':<15} {'Distance'}")
print("="*95)
for _, row in far_facilities.iterrows():
    print(f"{row['Name']:<40} {row['District']:<25} {row['Region']:<15} {row['distance_from_centroid_km']} km")

Facilities more than 50km from centroid: 73

Name                                     District                  Region          Distance
ABUADI CHPS                              ADAKLU                    VOLTA           301.761 km
DENYASE CHPS                             TWIFO ATI MORKWA          CENTRAL         193.508 km
WRCH                                     NANUMBA SOUTH             NORTHERN        121.62 km
DAS MEDICAL SERVICES                     NANUMBA SOUTH             NORTHERN        121.608 km
ZONYONI CHPS                             NANUMBA SOUTH             NORTHERN        121.603 km
YASHFAH MEDICAL SERVICES                 NANUMBA SOUTH             NORTHERN        121.517 km
DASHE CHPS                               NORTH-EAST GONJA          SAVANNAH        73.815 km
KOKOASE CHPS                             WASSA EAST                WESTERN         72.117 km
JANTONG HEALTH CENTRE                    NORTH-EAST GONJA          SAVANNAH        71.043 km
KOLIKOPE CHPS        

In [39]:
pip install geopandas


   ---------------------------------------- 0.0/22.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/22.9 MB ? eta -:--:--
   - -------------------------------------- 0.8/22.9 MB 2.6 MB/s eta 0:00:09
   - -------------------------------------- 1.0/22.9 MB 1.8 MB/s eta 0:00:13
   -- ------------------------------------- 1.3/22.9 MB 1.8 MB/s eta 0:00:12
   -- ------------------------------------- 1.3/22.9 MB 1.8 MB/s eta 0:00:12
   -- ------------------------------------- 1.3/22.9 MB 1.8 MB/s eta 0:00:12
   ---- ----------------------------------- 2.4/22.9 MB 1.8 MB/s eta 0:00:12
   ----- ---------------------------------- 3.1/22.9 MB 1.9 MB/s eta 0:00:11
   ------ --------------------------------- 3.7/22.9 MB 2.0 MB/s eta 0:00:10
   ------ --------------------------------- 3.9/22.9 MB 2.0 MB/s eta 0:00:10
   ------- -------------------------------- 4.2/22.9 MB 2.0 MB/s eta 0:00:10
   ------- -------------------------------- 4.5/22.9 MB 2.0 MB/s eta 0:00:10
   ------- -

In [40]:
#Reading the GeoJSON - Polygon data

import geopandas as gpd

# Load the region boundaries
regions_gdf = gpd.read_file(r"C:\Users\hp\Downloads\gadm41_GHA_1.json~1")

print(f"Shape: {regions_gdf.shape}")
print(f"\nColumns: {list(regions_gdf.columns)}")
print(f"\nRegion names:")
for name in sorted(regions_gdf['NAME_1'].unique()):
    print(f"  {name}")

DataSourceError: C:\Users\hp\Downloads\gadm41_GHA_1.json~1: Permission denied

In [41]:
import os

# Let's see what's actually in your Downloads folder matching "gadm"
for f in os.listdir(r"C:\Users\hp\Downloads"):
    if 'gadm' in f.lower() or 'gha' in f.lower():
        print(f)

AMLCFT Awareness Training for Hubtel Ghana - General Staff 11062025.pptx
Challenges and PainPoints of Insurance Industry in Ghana (3) (1).pdf
Challenges and PainPoints of Insurance Industry in Ghana (3) (1).pptx
Challenges and PainPoints of Insurance Industry in Ghana (3) (2).pdf
Challenges and PainPoints of Insurance Industry in Ghana (3) (3).pdf
Challenges and PainPoints of Insurance Industry in Ghana (3) (4).pdf
Challenges and PainPoints of Insurance Industry in Ghana (3).pdf
Client_Acquisition_Specialist_Smart_POS_Ghana.pdf
Crops by Production Value in Ghana (2005).csv
events passed with username and ghana card.xlsx
Exposing Data Analyst Salaries in Ghana (1).mp4
Exposing Data Analyst Salaries in Ghana (2).mp4
Exposing Data Analyst Salaries in Ghana (3).mp4
Exposing Data Analyst Salaries in Ghana.mp4
Female_Entrepreneurship_and_Ghanas_Infor.pdf
Final - Towards Equity_  A case study on the challenges, pain points and possible opportunities for women-owned businesses in Ghana.docx
ga

In [42]:
regions_gdf = gpd.read_file(r"C:\Users\hp\Downloads\gadm41_GHA_1.shp")

print(f"Shape: {regions_gdf.shape}")
print(f"\nColumns: {list(regions_gdf.columns)}")
print(f"\nRegion names:")
for name in sorted(regions_gdf['NAME_1'].unique()):
    print(f"  {name}")

Shape: (16, 12)

Columns: ['GID_1', 'GID_0', 'COUNTRY', 'NAME_1', 'VARNAME_1', 'NL_NAME_1', 'TYPE_1', 'ENGTYPE_1', 'CC_1', 'HASC_1', 'ISO_1', 'geometry']

Region names:
  Ahafo
  Ashanti
  Bono
  Bono East
  Central
  Eastern
  Greater Accra
  North East
  Northern
  Oti
  Savannah
  Upper East
  Upper West
  Volta
  Western
  Western North


In [43]:
# 16 regions! It's up to date. Now let's do the polygon validation — check every facility's coordinates against its stated region:

from shapely.geometry import Point

# Convert our facilities to a GeoDataFrame
facilities_gdf = gpd.GeoDataFrame(
    master,
    geometry=gpd.points_from_xy(master['Longitude'], master['Latitude']),
    crs="EPSG:4326"
)

# Make sure regions_gdf has the same CRS
regions_gdf = regions_gdf.to_crs("EPSG:4326")

# For each facility, check which region polygon it actually falls in
facilities_with_region = gpd.sjoin(facilities_gdf, regions_gdf[['NAME_1', 'geometry']], how='left', predicate='within')

# Compare stated region vs actual region
facilities_with_region['Region_upper'] = facilities_with_region['Region'].str.upper()
facilities_with_region['Actual_Region_upper'] = facilities_with_region['NAME_1'].str.upper()

# Find mismatches
mismatched = facilities_with_region[
    facilities_with_region['Region_upper'] != facilities_with_region['Actual_Region_upper']
]

# Also find facilities that didn't fall in ANY region (could be offshore/bad coordinates)
no_region = facilities_with_region[facilities_with_region['NAME_1'].isna()]

print(f"Total facilities: {len(master)}")
print(f"Coordinates in CORRECT region: {len(facilities_with_region) - len(mismatched) - len(no_region)}")
print(f"Coordinates in WRONG region: {len(mismatched)}")
print(f"Coordinates outside ALL regions: {len(no_region)}")

Total facilities: 9978
Coordinates in CORRECT region: 9889
Coordinates in WRONG region: 85
Coordinates outside ALL regions: 4


In [45]:
# Lets see the details!

# Show the mismatched facilities
print(f"=== FACILITIES WITH COORDINATES IN WRONG REGION ({len(mismatched)}) ===\n")
print(f"{'Name':<35} {'Stated Region':<18} {'Actual Region':<18} {'Distance_km'}")
print("="*95)

mismatch_display = mismatched[['Name', 'Region', 'NAME_1', 'distance_from_centroid_km']].sort_values('distance_from_centroid_km', ascending=False)

for _, row in mismatch_display.iterrows():
    print(f"{row['Name']:<35} {row['Region']:<18} {row['NAME_1']:<18} {row['distance_from_centroid_km']} km")

print(f"\n\n=== FACILITIES OUTSIDE ALL REGIONS ({len(no_region)}) ===\n")
for _, row in no_region.iterrows():
    print(f"  {row['Name']} | {row['District']} | {row['Region']} | Lat: {row['Latitude']}, Lon: {row['Longitude']}")

=== FACILITIES WITH COORDINATES IN WRONG REGION (85) ===

Name                                Stated Region      Actual Region      Distance_km
ABUADI CHPS                         VOLTA              Western            301.761 km
DENYASE CHPS                        CENTRAL            Bono               193.508 km
KOKOASE CHPS                        WESTERN            Central            72.117 km
KOLIKOPE CHPS                       EASTERN            Oti                70.59 km
FUU_KPENAYILI  CHPS                 SAVANNAH           Northern           67.281 km
MANCHARE CHPS                       EASTERN            Bono East          62.417 km
KIJEWU CHPS                         SAVANNAH           Bono East          55.265 km
KPAJAI NO. 2 CHPS                   NORTHERN           Savannah           52.369 km
LONTO HEALTH CENTRE                 NORTHERN           Savannah           47.347 km
LOLOTO CHPS                         NORTHERN           Savannah           40.606 km
ATUOBIKROM CHPS

In [46]:
# Let's look at all the stated vs actual region pairs
print("STATED REGION → ACTUAL REGION (count)")
print("="*60)

pairs = mismatched.groupby(['Region', 'NAME_1']).size().sort_values(ascending=False)
for (stated, actual), count in pairs.items():
    print(f"  {stated:<20} → {actual:<20} ({count})")

STATED REGION → ACTUAL REGION (count)
  OTI                  → Volta                (18)
  NORTHERN             → Savannah             (9)
  EASTERN              → Bono East            (6)
  EASTERN              → Ashanti              (5)
  CENTRAL              → Western              (4)
  VOLTA                → Greater Accra        (4)
  EASTERN              → Greater Accra        (3)
  GREATER ACCRA        → Eastern              (3)
  UPPER WEST           → Savannah             (3)
  EASTERN              → Central              (3)
  BONO EAST            → Oti                  (2)
  CENTRAL              → Greater Accra        (2)
  ASHANTI              → Bono East            (2)
  GREATER ACCRA        → Central              (2)
  CENTRAL              → Bono                 (1)
  BONO EAST            → Bono                 (1)
  BONO EAST            → Ashanti              (1)
  AHAFO                → Ashanti              (1)
  ASHANTI              → Ahafo                (1)
  BONO     

In [47]:
# Count by category

# 1. Old-to-new region split cases (Oti/Volta, Northern/Savannah, etc.)
old_new_splits = mismatched[
    ((mismatched['Region_upper'] == 'OTI') & (mismatched['Actual_Region_upper'] == 'VOLTA')) |
    ((mismatched['Region_upper'] == 'VOLTA') & (mismatched['Actual_Region_upper'] == 'OTI')) |
    ((mismatched['Region_upper'] == 'NORTHERN') & (mismatched['Actual_Region_upper'] == 'SAVANNAH')) |
    ((mismatched['Region_upper'] == 'SAVANNAH') & (mismatched['Actual_Region_upper'] == 'NORTHERN')) |
    ((mismatched['Region_upper'] == 'NORTHERN') & (mismatched['Actual_Region_upper'] == 'NORTH EAST')) |
    ((mismatched['Region_upper'] == 'ASHANTI') & (mismatched['Actual_Region_upper'] == 'AHAFO')) |
    ((mismatched['Region_upper'] == 'AHAFO') & (mismatched['Actual_Region_upper'] == 'ASHANTI')) |
    ((mismatched['Region_upper'] == 'ASHANTI') & (mismatched['Actual_Region_upper'] == 'BONO EAST')) |
    ((mismatched['Region_upper'] == 'BONO EAST') & (mismatched['Actual_Region_upper'] == 'ASHANTI')) |
    ((mismatched['Region_upper'] == 'BONO') & (mismatched['Actual_Region_upper'] == 'BONO EAST')) |
    ((mismatched['Region_upper'] == 'BONO EAST') & (mismatched['Actual_Region_upper'] == 'BONO')) |
    ((mismatched['Region_upper'] == 'WESTERN') & (mismatched['Actual_Region_upper'] == 'WESTERN NORTH')) |
    ((mismatched['Region_upper'] == 'WESTERN NORTH') & (mismatched['Actual_Region_upper'] == 'WESTERN')) |
    ((mismatched['Region_upper'] == 'BONO') & (mismatched['Actual_Region_upper'] == 'WESTERN NORTH')) |
    ((mismatched['Region_upper'] == 'WESTERN NORTH') & (mismatched['Actual_Region_upper'] == 'AHAFO')) |
    ((mismatched['Region_upper'] == 'BONO EAST') & (mismatched['Actual_Region_upper'] == 'OTI')) |
    ((mismatched['Region_upper'] == 'EASTERN') & (mismatched['Actual_Region_upper'] == 'BONO EAST')) |
    ((mismatched['Region_upper'] == 'EASTERN') & (mismatched['Actual_Region_upper'] == 'OTI')) |
    ((mismatched['Region_upper'] == 'NORTHERN') & (mismatched['Actual_Region_upper'] == 'OTI')) |
    ((mismatched['Region_upper'] == 'UPPER WEST') & (mismatched['Actual_Region_upper'] == 'SAVANNAH')) |
    ((mismatched['Region_upper'] == 'SAVANNAH') & (mismatched['Actual_Region_upper'] == 'BONO EAST')) |
    ((mismatched['Region_upper'] == 'SAVANNAH') & (mismatched['Actual_Region_upper'] == 'BONO'))
]

# 2. Neighboring region boundary cases (not from splits, but close neighbors)
remaining = mismatched[~mismatched.index.isin(old_new_splits.index)]
boundary_cases = remaining[remaining['distance_from_centroid_km'] < 25]

# 3. Genuinely bad coordinates
bad_coords = remaining[remaining['distance_from_centroid_km'] >= 25]

# 4. Outside Ghana
outside = len(no_region)

print(f"=== BREAKDOWN OF MISMATCHED FACILITIES ===\n")
print(f"1. Region split cases (old map vs new map):  {len(old_new_splits)}")
print(f"2. Neighboring region boundary cases:        {len(boundary_cases)}")
print(f"3. Genuinely bad coordinates:                {len(bad_coords)}")
print(f"4. Outside Ghana entirely:                   {outside}")
print(f"                                             --------")
print(f"   TOTAL:                                    {len(old_new_splits) + len(boundary_cases) + len(bad_coords) + outside}")

=== BREAKDOWN OF MISMATCHED FACILITIES ===

1. Region split cases (old map vs new map):  51
2. Neighboring region boundary cases:        27
3. Genuinely bad coordinates:                7
4. Outside Ghana entirely:                   4
                                             --------
   TOTAL:                                    89


In [48]:
# Update the stated region to match the actual polygon region for the 51 split cases

# Get the IDs of the region split facilities
split_ids = old_new_splits['ID'].values

# Create a lookup: ID -> actual region from polygon
actual_region_lookup = dict(zip(
    facilities_with_region['ID'], 
    facilities_with_region['NAME_1'].str.upper()
))

# Count before
print("BEFORE:")
changed = []
for fid in split_ids:
    old_region = master.loc[master['ID'] == fid, 'Region'].values[0]
    new_region = actual_region_lookup.get(fid)
    if new_region and old_region != new_region:
        changed.append((master.loc[master['ID'] == fid, 'Name'].values[0], old_region, new_region))

for name, old, new in changed:
    print(f"  {name:<40} {old:<18} → {new}")

# Apply the update
for fid in split_ids:
    new_region = actual_region_lookup.get(fid)
    if new_region:
        master.loc[master['ID'] == fid, 'Region'] = new_region

print(f"\n✅ Updated {len(changed)} facilities to their correct 2019 region")
print(f"\nRegion counts after update:")
print(master['Region'].value_counts().to_string())

BEFORE:
  AHENBRONUM (AFRANCHO) CHPS               ASHANTI            → BONO EAST
  AKPAFU ADORKOR HEALTH CENTRE             OTI                → VOLTA
  AKPAFU MEMPEASEM HEALTH CENTRE           OTI                → VOLTA
  AKPAFU ODOMI CHPS                        OTI                → VOLTA
  AKPAFU TODZI CHPS                        OTI                → VOLTA
  AKURAFU CHPS                             WESTERN NORTH      → AHAFO
  ASAKYIRI CHPS                            OTI                → VOLTA
  AWIAKROM CHPS                            BONO               → WESTERN NORTH
  BAKPA CHPS ZONE                          BONO EAST          → OTI
  DASAGUA CHPS                             BONO EAST          → ASHANTI
  DIGYA CHPS                               EASTERN            → BONO EAST
  DOLLAR POWER CHPS                        SAVANNAH           → BONO
  DOMANGYLI CHPS                           UPPER WEST         → SAVANNAH
  FUU_KPENAYILI  CHPS                      SAVANNAH           → 

In [49]:
# Show the 27 boundary cases in detail
print(f"=== 27 NEIGHBORING REGION BOUNDARY CASES ===\n")
print(f"{'Name':<40} {'District':<25} {'Stated Region':<18} {'Actual Region':<18} {'Dist_km'}")
print("="*115)

for _, row in boundary_cases.sort_values('distance_from_centroid_km', ascending=False).iterrows():
    print(f"{row['Name']:<40} {row['District']:<25} {row['Region']:<18} {row['NAME_1']:<18} {row['distance_from_centroid_km']}")

=== 27 NEIGHBORING REGION BOUNDARY CASES ===

Name                                     District                  Stated Region      Actual Region      Dist_km
BONKROM CHPS                             KWAHU AFRAM PLAINS SOUTH  EASTERN            Ashanti            24.621
AWULAE BLAY IV CHPS                      ELLEMBELLE                WESTERN            nan                23.8
MERCIFUL HOSPITAL                        NORTH TONGU               VOLTA              Eastern            22.113
DEDUKOPE CHPS                            NORTH TONGU               VOLTA              Greater Accra      19.814
AKENKENSU CHPS                           ACHIASE                   EASTERN            Central            18.905
AKOSOMBO CHPS                            ACHIASE                   EASTERN            Central            17.88
MARGO MARTERNITY HOME                    AWUTU SENYA               CENTRAL            Greater Accra      17.189
AMENASE CHPS                             UPPER DENKYIRA WEST

In [50]:
# Show the 7 bad coordinates + 4 outside Ghana
print(f"=== 7 GENUINELY BAD COORDINATES ===\n")
print(f"{'Name':<35} {'District':<25} {'Stated Region':<18} {'Actual Region':<18} {'Dist_km'}")
print("="*110)

for _, row in bad_coords.sort_values('distance_from_centroid_km', ascending=False).iterrows():
    print(f"{row['Name']:<35} {row['District']:<25} {row['Region']:<18} {row['NAME_1']:<18} {row['distance_from_centroid_km']}")

print(f"\n\n=== 4 OUTSIDE GHANA ===\n")
for _, row in no_region.iterrows():
    print(f"  {row['Name']:<35} {row['District']:<25} {row['Region']:<18} Lat: {row['Latitude']}, Lon: {row['Longitude']}")

=== 7 GENUINELY BAD COORDINATES ===

Name                                District                  Stated Region      Actual Region      Dist_km
ABUADI CHPS                         ADAKLU                    VOLTA              Western            301.761
DENYASE CHPS                        TWIFO ATI MORKWA          CENTRAL            Bono               193.508
KOKOASE CHPS                        WASSA EAST                WESTERN            Central            72.117
ATUOBIKROM CHPS                     KWAHU SOUTH               EASTERN            Ashanti            39.12
BESEASE CHPS                        KWAHU SOUTH               EASTERN            Ashanti            38.754
DOME CHPS                           KWAHU AFRAM PLAINS SOUTH  EASTERN            Ashanti            36.458
NSUOGYASO CHPS                      KWAHU AFRAM PLAINS SOUTH  EASTERN            Ashanti            35.337


=== 4 OUTSIDE GHANA ===

  AWULAE BLAY IV CHPS                 ELLEMBELLE                WESTERN       

In [51]:
corrections = {
    'DENYASE CHPS': (5.854229606029756, -1.8013720077775004),
    'KOKOASE CHPS': (5.607065109510294, -2.487296905928909),
    'ATUOBIKROM CHPS': (6.639825479861686, -0.9623284461520769),
    'BESEASE CHPS': (6.041993555712988, -2.0105370635976705),
    'DOME CHPS': (7.395566021472596, -0.6246518639064519),
    'NSUOGYASO CHPS': (7.624459860903312, -0.49406561717014935),
    'BOATYARD CHPS': (5.287440475229086, -0.7292129482592253),
    'GONOKROM CHPS': (7.24, -2.96),
}

for name, (lat, lon) in corrections.items():
    master.loc[master['Name'] == name, 'Latitude'] = lat
    master.loc[master['Name'] == name, 'Longitude'] = lon

# Recalculate distances
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

for name in corrections.keys():
    row = master[master['Name'] == name].iloc[0]
    dist = haversine(row['Latitude'], row['Longitude'], row['Centroid_Lat'], row['Centroid_Lon'])
    master.loc[master['Name'] == name, 'distance_from_centroid_km'] = round(dist, 3)

# Show updated values
print("Updated facilities:\n")
print(f"{'Name':<25} {'New Lat':<15} {'New Lon':<15} {'New Distance_km'}")
print("="*70)
for name in corrections.keys():
    row = master[master['Name'] == name].iloc[0]
    print(f"{name:<25} {row['Latitude']:<15.4f} {row['Longitude']:<15.4f} {row['distance_from_centroid_km']}")

# Flag only the 3 we truly couldn't find
unfound = ['ABUADI CHPS', 'AWULAE BLAY IV CHPS', 'KEDZIKOPE CHPS']
master['bad_coordinates'] = master['Name'].isin(unfound)

print(f"\nFixed: 8 facilities")
print(f"Still unfound (flagged): {master['bad_coordinates'].sum()}")

Updated facilities:

Name                      New Lat         New Lon         New Distance_km
DENYASE CHPS              5.8542          -1.8014         29.999
KOKOASE CHPS              5.6071          -2.4873         95.956
ATUOBIKROM CHPS           6.6398          -0.9623         43.206
BESEASE CHPS              6.0420          -2.0105         204.522
DOME CHPS                 7.3956          -0.6247         236.09
NSUOGYASO CHPS            7.6245          -0.4941         82.288
BOATYARD CHPS             5.2874          -0.7292         15.622
GONOKROM CHPS             7.2400          -2.9600         10.875

Fixed: 8 facilities
Still unfound (flagged): 3


In [52]:
# Let's see the districts and centroids for the problematic ones
problem_names = ['BESEASE CHPS', 'DOME CHPS', 'KOKOASE CHPS', 'ATUOBIKROM CHPS', 'NSUOGYASO CHPS']

print(f"{'Name':<25} {'District':<30} {'Fac Lat':<12} {'Fac Lon':<12} {'Cent Lat':<12} {'Cent Lon':<12} {'Dist_km'}")
print("="*115)

for name in problem_names:
    row = master[master['Name'] == name].iloc[0]
    print(f"{name:<25} {row['District']:<30} {row['Latitude']:<12.4f} {row['Longitude']:<12.4f} {row['Centroid_Lat']:<12.4f} {row['Centroid_Lon']:<12.4f} {row['distance_from_centroid_km']}")

Name                      District                       Fac Lat      Fac Lon      Cent Lat     Cent Lon     Dist_km
BESEASE CHPS              AKWAPIM NORTH                  6.0420       -2.0105      5.9559       -0.1631      204.522
DOME CHPS                 ASUNAFO NORTH                  7.3956       -0.6247      6.8226       -2.6849      236.09
KOKOASE CHPS              WASSA EAST                     5.6071       -2.4873      5.3483       -1.6603      95.956
ATUOBIKROM CHPS           KWAHU SOUTH                    6.6398       -0.9623      6.6323       -0.5712      43.206
NSUOGYASO CHPS            KWAHU AFRAM PLAINS SOUTH       7.6245       -0.4941      6.8955       -0.3653      82.288


In [53]:
# Load district-level polygons
districts_gdf = gpd.read_file(r"C:\Users\hp\Downloads\gadm41_GHA_2.shp")

print(f"Shape: {districts_gdf.shape}")
print(f"\nColumns: {list(districts_gdf.columns)}")
print(f"\nNumber of districts: {districts_gdf['NAME_2'].nunique()}")
print(f"\nFirst 10 district names:")
for name in sorted(districts_gdf['NAME_2'].unique())[:10]:
    print(f"  {name}")

Shape: (260, 14)

Columns: ['GID_2', 'GID_0', 'COUNTRY', 'GID_1', 'NAME_1', 'NL_NAME_1', 'NAME_2', 'VARNAME_2', 'NL_NAME_2', 'TYPE_2', 'ENGTYPE_2', 'CC_2', 'HASC_2', 'geometry']

Number of districts: 260

First 10 district names:
  Ablekuma Central
  Ablekuma North
  Ablekuma West
  Abuakwa North
  Abuakwa South
  Abura-Asebu-Kwamankese
  Accra
  Achiase
  Ada East
  Ada West


In [54]:
# Compare district names
gadm_districts = set(districts_gdf['NAME_2'].str.upper().str.strip())
our_districts = set(master['District'].str.upper().str.strip())

print(f"GADM districts: {len(gadm_districts)}")
print(f"Our districts: {len(our_districts)}")

missing_from_gadm = our_districts - gadm_districts
print(f"\nIn our data but not in GADM ({len(missing_from_gadm)}):")
for d in sorted(missing_from_gadm):
    print(f"  {d}")

in_gadm_not_ours = gadm_districts - our_districts
print(f"\nIn GADM but not in our data ({len(in_gadm_not_ours)}):")
for d in sorted(in_gadm_not_ours):
    print(f"  {d}")

GADM districts: 260
Our districts: 261

In our data but not in GADM (57):
  ACCRA METRO
  ADENTAN
  AFADJATO SOUTH
  AFIGYA KWABRE NORTH
  AFIGYA KWABRE SOUTH
  AGORTIME-ZIOPE
  AHAFO ANO NORTH
  AHAFO ANO SOUTH EAST
  AHAFO ANO SOUTH WEST
  AKWAPIM NORTH
  AKWAPIM SOUTH
  ASANTE AKIM CENTRAL
  ASANTE AKIM NORTH
  ASANTE AKIM SOUTH
  ASANTE MAMPONG
  ASENE MANSO AKROSO
  ASOKORE MAMPONG
  ATEBUBU-AMANTEN
  ATWIMA KWANWOMA
  ATWIMA MPONUA
  ATWIMA NWABIAGYA
  ATWIMA NWABIAGYA NORTH
  AWUTU SENYA
  BAWKU MUNICIPAL
  BEREKUM
  BIBIANI-ANHWIASO-BEKWAI
  BOLGATANGA EAST
  BOLGATANGA MUNICIPAL
  BUNKPURUGU -NAKPANDURI
  DORMAA MUNICIPAL
  EFFIA-KWESIMINTSIM
  EFUTU
  GUAN
  GUSHIEGU
  HO
  KASENA-NANKANA
  KASENA-NANKANA WEST
  KETA
  KOMENDA-EDNA-EGUAFO-ABIREM
  LAMBUSSIE
  MAMPRUGU-MOAGDURI
  NINGO PRAMPRAM
  NORTH-EAST GONJA
  NSAWAM-ADOAGYIRI
  OKAI KOI NORTH
  SAGNARIGU
  SEKONDI-TAKORADI
  SEKYERE AFRAM PLAINS
  SHAI-OSUDOKU
  SUNYANI MUNICIPAL
  TARKWA-NSUAEM
  TATALE-SANGULE
  TECHIM

In [55]:
# Apply simple_key to both datasets to match
districts_gdf['simple_key'] = districts_gdf['NAME_2'].apply(make_simple_key)
master['simple_key'] = master['District'].apply(make_simple_key)

gadm_keys = set(districts_gdf['simple_key'])
our_keys = set(master['simple_key'])

still_missing = our_keys - gadm_keys
still_extra = gadm_keys - our_keys

print(f"Matched after simple_key: {len(our_keys & gadm_keys)} of {len(our_keys)}")

if still_missing:
    print(f"\nStill unmatched from our data ({len(still_missing)}):")
    for k in sorted(still_missing):
        print(f"  {k}")
        
if still_extra:
    print(f"\nStill unmatched from GADM ({len(still_extra)}):")
    for k in sorted(still_extra):
        print(f"  {k}")

Matched after simple_key: 237 of 261

Still unmatched from our data (24):
  ACCRA METRO
  ADENTAN
  AFADJATO SOUTH
  AGORTIME ZIOPE
  AKWAPIM NORTH
  AKWAPIM SOUTH
  ASANTE MAMPONG
  ATEBUBU AMANTEN
  ATWIMA NWABIAGYA
  AWUTU SENYA
  BEREKUM
  BOLGATANGA EAST
  EFUTU
  GUAN
  GUSHIEGU
  KASENA NANKANA
  KOMENDA EDNA EGUAFO ABIREM
  LAMBUSSIE
  OKAI KOI NORTH
  SAGNARIGU
  SEKYERE AFRAM PLAINS
  TATALE SANGULE
  TWIFO ATI MORKWA
  UPPER MANYA KROBO

Still unmatched from GADM (23):
  ACCRA
  ADENTA
  AFADZATO SOUTH
  AGOTIME ZIOPE
  AKWAPEM NORTH
  AKWAPEM SOUTH
  ATEBUBU AMANTIN
  ATWIMA NWABIAGYA SOUTH
  AWUTU SENYA WEST
  BEREKUM EAST
  BOLGA EAST
  EFFUTU
  GUSHEGU
  KASENA NANKANA EAST
  KOMENDA EDINA EGUAFO ABIREM
  LAMBUSSIE KARNI
  MAMPONG
  OKAIKWEI NORTH
  SAGNERIGU
  SEKYERE AFRAM PLAINS NORTH
  TATALE SANGULI
  TWIFO ATTI MORKWA
  UPPER MANYA


In [56]:
# Manual mapping: our district simple_key -> GADM simple_key
district_gadm_mapping = {
    'ACCRA METRO': 'ACCRA',
    'ADENTAN': 'ADENTA',
    'AFADJATO SOUTH': 'AFADZATO SOUTH',
    'AGORTIME ZIOPE': 'AGOTIME ZIOPE',
    'AKWAPIM NORTH': 'AKWAPEM NORTH',
    'AKWAPIM SOUTH': 'AKWAPEM SOUTH',
    'ASANTE MAMPONG': 'MAMPONG',
    'ATEBUBU AMANTEN': 'ATEBUBU AMANTIN',
    'ATWIMA NWABIAGYA': 'ATWIMA NWABIAGYA SOUTH',
    'AWUTU SENYA': 'AWUTU SENYA WEST',
    'BEREKUM': 'BEREKUM EAST',
    'BOLGATANGA EAST': 'BOLGA EAST',
    'EFUTU': 'EFFUTU',
    'GUSHIEGU': 'GUSHEGU',
    'KASENA NANKANA': 'KASENA NANKANA EAST',
    'KOMENDA EDNA EGUAFO ABIREM': 'KOMENDA EDINA EGUAFO ABIREM',
    'LAMBUSSIE': 'LAMBUSSIE KARNI',
    'OKAI KOI NORTH': 'OKAIKWEI NORTH',
    'SAGNARIGU': 'SAGNERIGU',
    'SEKYERE AFRAM PLAINS': 'SEKYERE AFRAM PLAINS NORTH',
    'TATALE SANGULE': 'TATALE SANGULI',
    'TWIFO ATI MORKWA': 'TWIFO ATTI MORKWA',
    'UPPER MANYA KROBO': 'UPPER MANYA',
}

# That's 23 mappings but we have 24 unmatched — GUAN is missing
# Check if GUAN exists in GADM under a different name
print("Searching for GUAN in GADM...")
for name in sorted(districts_gdf['NAME_2'].unique()):
    if 'GUAN' in name.upper():
        print(f"  Found: {name}")

# If not found, let's see what's left in GADM that we haven't matched
matched_gadm = set(district_gadm_mapping.values()) | (our_keys & gadm_keys)
remaining_gadm = gadm_keys - matched_gadm
print(f"\nUnmatched GADM districts after mapping:")
for k in sorted(remaining_gadm):
    print(f"  {k}")

Searching for GUAN in GADM...

Unmatched GADM districts after mapping:


In [57]:
# What do we know about GUAN in our data?
guan = master[master['District'] == 'GUAN']
print(f"Facilities in GUAN: {len(guan)}")
print(f"Region: {guan['Region'].iloc[0]}")
print(f"Sample coordinates: Lat {guan['Latitude'].iloc[0]:.4f}, Lon {guan['Longitude'].iloc[0]:.4f}")

# Let's see which GADM district these coordinates fall in
from shapely.geometry import Point

sample_point = Point(guan['Longitude'].iloc[0], guan['Latitude'].iloc[0])
for _, row in districts_gdf.iterrows():
    if row['geometry'].contains(sample_point):
        print(f"\nGUAN coordinates fall inside GADM district: {row['NAME_2']} ({row['NAME_1']})")
        break

Facilities in GUAN: 17
Region: VOLTA
Sample coordinates: Lat 7.2813, Lon 0.5281

GUAN coordinates fall inside GADM district: Hohoe (Volta)


In [58]:
districts_gdf2 = gpd.read_file(r"C:\Users\hp\Downloads\geoBoundaries-GHA-ADM2.geojson")

print(f"Shape: {districts_gdf2.shape}")
print(f"\nColumns: {list(districts_gdf2.columns)}")
print(f"\nNumber of districts: {districts_gdf2['shapeName'].nunique()}")

# Check if GUAN is in there
print(f"\nSearching for GUAN:")
for name in sorted(districts_gdf2['shapeName'].unique()):
    if 'GUAN' in name.upper() or 'guan' in name.lower():
        print(f"  Found: {name}")

print(f"\nFirst 15 district names:")
for name in sorted(districts_gdf2['shapeName'].unique())[:15]:
    print(f"  {name}")

Shape: (260, 6)

Columns: ['shapeName', 'shapeISO', 'shapeID', 'shapeGroup', 'shapeType', 'geometry']

Number of districts: 260

Searching for GUAN:

First 15 district names:
  Ablekuma Central Municipal
  Ablekuma North Municipal
  Ablekuma West Municipal
  Abuakwa North
  Abuakwa South
  Abura-asebu-kwamankese
  Accra Metropolis
  Achiase
  Ada East
  Ada West
  Adaklu
  Adansi Akrofuom
  Adansi Asokwa
  Adansi North
  Adansi South


In [59]:
guan_facilities = master[master['District'] == 'GUAN']
print(f"Total GUAN facilities: {len(guan_facilities)}\n")
print(f"{'Name':<45} {'Sub-District':<25} {'Community'}")
print("="*95)
for _, row in guan_facilities.iterrows():
    print(f"{row['Name']:<45} {row['Sub-District']:<25} {row['Community']}")

Total GUAN facilities: 17

Name                                          Sub-District              Community
AKPAFU ADORKOR HEALTH CENTRE                  AKPAFU                    AKPAFU ADORKOR
AKPAFU MEMPEASEM HEALTH CENTRE                AKPAFU                    MEMPEASEM
AKPAFU ODOMI CHPS                             AKPAFU                    ODOMI
AKPAFU TODZI CHPS                             AKPAFU                    TODZI
LIKPE ABRANI   HEALTH CENTRE                  LIKPE ABRANI              LIKPE ABRANI
LIKPE AGBOZUME CHPS                           LIKPE ABRANI              LIKPE AGBOZOME
LIKPE BAKUA POLYCLINIC                        LIKPE BAKWA               LIKPE BAKWA
LIKPE BALA HEALTH CENTRE                      LIKPE BAKWA               LIKPE BALA
LIKPE KOFORIDUA CHPS                          LIKPE ABRANI              LIKPE KOFORIDUA
LIKPE KUKURANTUMI CHPS                        LIKPE ABRANI              LIKPE KUKURANTUMI
LIKPE MATE CHPS                               LIK

In [60]:
guan_gdf = gpd.GeoDataFrame(
    guan_facilities,
    geometry=gpd.points_from_xy(guan_facilities['Longitude'], guan_facilities['Latitude']),
    crs="EPSG:4326"
)

guan_check = gpd.sjoin(guan_gdf, districts_gdf[['NAME_2', 'geometry']], how='left', predicate='within')

print(f"Where GUAN facilities fall on the old map:\n")
print(f"{'Name':<45} {'Old map district'}")
print("="*65)
for _, row in guan_check.iterrows():
    print(f"{row['Name']:<45} {row['NAME_2']}")

print(f"\nSummary:")
print(guan_check['NAME_2'].value_counts().to_string())

Where GUAN facilities fall on the old map:

Name                                          Old map district
AKPAFU ADORKOR HEALTH CENTRE                  Hohoe
AKPAFU MEMPEASEM HEALTH CENTRE                Hohoe
AKPAFU ODOMI CHPS                             Hohoe
AKPAFU TODZI CHPS                             Hohoe
LIKPE ABRANI   HEALTH CENTRE                  Hohoe
LIKPE AGBOZUME CHPS                           Hohoe
LIKPE BAKUA POLYCLINIC                        Hohoe
LIKPE BALA HEALTH CENTRE                      Hohoe
LIKPE KOFORIDUA CHPS                          Hohoe
LIKPE KUKURANTUMI CHPS                        Hohoe
LIKPE MATE CHPS                               Hohoe
LIKPE NKWANTA CHPS                            Hohoe
LOLOBI ASHIAMBI CHPS                          Hohoe
LOLOBI HUYEASEN CHPS                          Hohoe
LOLOBI KUMASI HEALTH CENTRE                   Hohoe
SANTROKOFI BENUA HEALTH CENTRE                Hohoe
SANTROKOFI BUME HEALTH CENTRE                 Hohoe

Summary:

In [61]:
from shapely.geometry import MultiPoint
from shapely.ops import unary_union

# Get all GUAN facility points
guan_points = MultiPoint(list(zip(guan_facilities['Longitude'], guan_facilities['Latitude'])))

# Create a convex hull (rubber band around the points)
guan_hull = guan_points.convex_hull

# Add a small buffer (roughly 2km) so the boundary isn't too tight
# 0.02 degrees is roughly 2km near the equator
guan_polygon = guan_hull.buffer(0.02)

# Cut GUAN out of Hohoe
hohoe_row = districts_gdf[districts_gdf['NAME_2'] == 'Hohoe'].iloc[0]
hohoe_old = hohoe_row['geometry']
hohoe_new = hohoe_old.difference(guan_polygon)

# Update Hohoe to the smaller version
districts_gdf.loc[districts_gdf['NAME_2'] == 'Hohoe', 'geometry'] = hohoe_new

# Add GUAN as a new row
guan_row = hohoe_row.copy()
guan_row['NAME_2'] = 'Guan'
guan_row['NAME_1'] = 'Oti'
guan_row['geometry'] = guan_polygon

districts_gdf = pd.concat([districts_gdf, gpd.GeoDataFrame([guan_row])], ignore_index=True)

print(f"Districts in shapefile now: {len(districts_gdf)}")
print(f"GUAN polygon area: {round(guan_polygon.area * 12321, 1)} approx sq km")

# Verify: check GUAN facilities again
guan_check2 = gpd.sjoin(guan_gdf, districts_gdf[['NAME_2', 'geometry']], how='left', predicate='within')
print(f"\nGUAN facilities now falling in:")
print(guan_check2['NAME_2'].value_counts().to_string())

Districts in shapefile now: 261
GUAN polygon area: 263.8 approx sq km

GUAN facilities now falling in:
NAME_2
Guan    17


C:\Users\hp\anaconda3\Lib\site-packages\geopandas\array.py:1754: UserWarning: CRS not set for some of the concatenation inputs. Setting output's CRS as WGS 84 (the single non-null crs provided).
  return GeometryArray(data, crs=_get_common_crs(to_concat))


In [62]:
print(f"Districts in shapefile now: {len(districts_gdf)}")

# Verify GUAN facilities
guan_check2 = gpd.sjoin(guan_gdf, districts_gdf[['NAME_2', 'geometry']], how='left', predicate='within')
print(f"\nGUAN facilities now falling in:")
print(guan_check2['NAME_2'].value_counts().to_string())

Districts in shapefile now: 261

GUAN facilities now falling in:
NAME_2
Guan    17


In [63]:
# Full district-level validation

# Create simple keys for matching
districts_gdf['simple_key'] = districts_gdf['NAME_2'].apply(make_simple_key)

# Map our hospital district names to GADM names
master['gadm_key'] = master['simple_key'].map(lambda x: district_gadm_mapping.get(x, x))

# Build the spatial join
facilities_gdf = gpd.GeoDataFrame(
    master,
    geometry=gpd.points_from_xy(master['Longitude'], master['Latitude']),
    crs="EPSG:4326"
)

districts_gdf = districts_gdf.to_crs("EPSG:4326")

facilities_district_check = gpd.sjoin(
    facilities_gdf, 
    districts_gdf[['NAME_2', 'simple_key', 'geometry']], 
    how='left', 
    predicate='within'
)

# Compare: does the facility's stated district match where it actually falls?
facilities_district_check['actual_district_key'] = facilities_district_check['simple_key_right']
facilities_district_check['district_match'] = (
    facilities_district_check['gadm_key'] == facilities_district_check['actual_district_key']
)

matched = facilities_district_check['district_match'].sum()
outside = facilities_district_check['NAME_2'].isna().sum()
wrong = (~facilities_district_check['district_match']).sum() - outside

print(f"=== DISTRICT LEVEL VALIDATION ===\n")
print(f"Total facilities:        {len(facilities_district_check)}")
print(f"In CORRECT district:     {matched} ({round(matched/len(master)*100, 1)}%)")
print(f"In WRONG district:       {wrong} ({round(wrong/len(master)*100, 1)}%)")
print(f"Outside all districts:   {outside}")

=== DISTRICT LEVEL VALIDATION ===

Total facilities:        9980
In CORRECT district:     8946 (89.7%)
In WRONG district:       1030 (10.3%)
Outside all districts:   4


In [64]:
# First fix the 2 extra rows
print(f"Total rows: {len(facilities_district_check)}")
print(f"Expected: {len(master)}")
dupes = facilities_district_check[facilities_district_check.duplicated(subset='ID', keep=False)]
print(f"Duplicate IDs: {len(dupes)}")

# Now look at the 1,030 wrong district facilities
wrong_districts = facilities_district_check[
    (~facilities_district_check['district_match']) & 
    (facilities_district_check['NAME_2'].notna())
]

# How far are they from their stated district centroid?
print(f"\nDistance from centroid for WRONG district facilities:")
print(wrong_districts['distance_from_centroid_km'].describe())

print(f"\nDistance from centroid for CORRECT district facilities:")
correct = facilities_district_check[facilities_district_check['district_match']]
print(correct['distance_from_centroid_km'].describe())

# How many are close (likely boundary cases)?
print(f"\nWrong district facilities by distance:")
for threshold in [5, 10, 15, 20, 30, 50]:
    count = len(wrong_districts[wrong_districts['distance_from_centroid_km'] <= threshold])
    print(f"  Within {threshold}km of stated centroid: {count} ({round(count/len(wrong_districts)*100, 1)}%)")

Total rows: 9980
Expected: 9978
Duplicate IDs: 4

Distance from centroid for WRONG district facilities:
count    1030.000000
mean       16.615164
std        24.992428
min         0.679000
25%         4.872750
50%        10.677000
75%        18.664250
max       301.761000
Name: distance_from_centroid_km, dtype: float64

Distance from centroid for CORRECT district facilities:
count    8946.000000
mean       11.224811
std         8.943209
min         0.048000
25%         4.906000
50%         9.295500
75%        15.229750
max       204.522000
Name: distance_from_centroid_km, dtype: float64

Wrong district facilities by distance:
  Within 5km of stated centroid: 269 (26.1%)
  Within 10km of stated centroid: 486 (47.2%)
  Within 15km of stated centroid: 659 (64.0%)
  Within 20km of stated centroid: 805 (78.2%)
  Within 30km of stated centroid: 902 (87.6%)
  Within 50km of stated centroid: 979 (95.0%)


In [65]:
# For the 1,030 wrong district facilities
# Show: stated district, actual district (from polygon), and how often each pair appears

wrong_districts = facilities_district_check[
    (~facilities_district_check['district_match']) & 
    (facilities_district_check['NAME_2'].notna())
]

pairs = wrong_districts.groupby(['District', 'NAME_2']).size().reset_index(name='count')
pairs = pairs.sort_values('count', ascending=False)

print(f"Total mismatched district pairs: {len(pairs)}")
print(f"\nTop 30 most common mismatches:\n")
print(f"{'Stated District':<30} {'Actual District (map)':<30} {'Count'}")
print("="*70)
for _, row in pairs.head(30).iterrows():
    print(f"{row['District']:<30} {row['NAME_2']:<30} {row['count']}")

Total mismatched district pairs: 303

Top 30 most common mismatches:

Stated District                Actual District (map)          Count
GA SOUTH                       Weija Gbawe                    44
EAST GONJA                     North East Gonja               33
ACCRA METRO                    Ablekuma West                  25
KUMASI                         Kwadaso                        25
KUMASI                         Old Tafo                       24
ACCRA METRO                    Ablekuma North                 24
GA WEST                        Ga North                       21
AWUTU SENYA EAST               Gomoa East                     21
JUABEN                         Ejisu                          20
ACCRA METRO                    Ablekuma Central               18
OBUASI EAST                    Obuasi                         17
ATWIMA NWABIAGYA NORTH         Atwima-Nwabiagya South         17
GA NORTH                       Ga West                        16
GOMOA CENTRAL    

In [66]:
# Check: do the polygon district names match our population data?
# Take the top mismatched pairs and see if the "actual" district exists in population data

pop_districts = set(population['District'].str.upper().str.strip())

print("Do the polygon's 'new' districts exist in our population data?\n")
top_pairs = pairs.head(15)
for _, row in top_pairs.iterrows():
    actual = row['NAME_2'].upper().strip()
    exists = actual in pop_districts or make_simple_key(actual) in set(population['District'].apply(make_simple_key))
    print(f"  {row['NAME_2']:<30} {'✅ YES' if exists else '❌ NO'}")

Do the polygon's 'new' districts exist in our population data?

  Weija Gbawe                    ✅ YES
  North East Gonja               ✅ YES
  Ablekuma West                  ✅ YES
  Kwadaso                        ✅ YES
  Old Tafo                       ✅ YES
  Ablekuma North                 ✅ YES
  Ga North                       ✅ YES
  Gomoa East                     ✅ YES
  Ejisu                          ❌ NO
  Ablekuma Central               ✅ YES
  Obuasi                         ✅ YES
  Atwima-Nwabiagya South         ✅ YES
  Ga West                        ✅ YES
  Gomoa East                     ✅ YES
  Keta Municipal                 ✅ YES


In [67]:
hospital_districts = master['District'].str.upper().str.strip().nunique()
polygon_districts = districts_gdf['NAME_2'].str.upper().str.strip().nunique()

print(f"Unique districts in hospital data: {hospital_districts}")
print(f"Unique districts in polygon data: {polygon_districts}")

Unique districts in hospital data: 261
Unique districts in polygon data: 261


In [68]:
# Let's look at the GA SOUTH -> Weija Gbawe case specifically
# Where are these 44 facilities sitting?

ga_south_wrong = wrong_districts[
    (wrong_districts['District'] == 'GA SOUTH') & 
    (wrong_districts['NAME_2'] == 'Weija Gbawe')
]

ga_south_correct = facilities_district_check[
    (facilities_district_check['District'] == 'GA SOUTH') & 
    (facilities_district_check['district_match'] == True)
]

print(f"GA SOUTH facilities in correct polygon: {len(ga_south_correct)}")
print(f"GA SOUTH facilities landing in Weija Gbawe: {len(ga_south_wrong)}")
print(f"Total GA SOUTH facilities: {len(ga_south_correct) + len(ga_south_wrong)}")

# Are there also facilities labeled WEIJA-GBAWE in our hospital data?
weija_facilities = master[master['District'].str.upper().str.contains('WEIJA')]
print(f"\nFacilities labeled as WEIJA-GBAWE in hospital data: {len(weija_facilities)}")

GA SOUTH facilities in correct polygon: 12
GA SOUTH facilities landing in Weija Gbawe: 44
Total GA SOUTH facilities: 56

Facilities labeled as WEIJA-GBAWE in hospital data: 50


In [69]:
# Step 1: Create district centroid fact table

# Start with our existing centroid data
centroids = all_sheets['district-centroids'].copy()
centroids = centroids.rename(columns={'Latitude': 'Centroid_Lat', 'Longitude': 'Centroid_Lon'})

# Create fact table with unique districts and centroids
fact_table = centroids[['District', 'Centroid_Lat', 'Centroid_Lon']].drop_duplicates()

# Add GUAN centroid (7°12'16.2"N = 7.2045, 0°31'8.8"E = 0.5191)
guan_centroid = pd.DataFrame({
    'District': ['Guan'],
    'Centroid_Lat': [7.2045],
    'Centroid_Lon': [0.5191]
})
fact_table = pd.concat([fact_table, guan_centroid], ignore_index=True)

print(f"Fact table: {len(fact_table)} districts")
print(f"\nSample:")
print(fact_table.head(10).to_string())
print(f"\nGUAN entry:")
print(fact_table[fact_table['District'] == 'Guan'].to_string())

Fact table: 262 districts

Sample:
          District  Centroid_Lat  Centroid_Lon
0    Asunafo North      6.822567     -2.684927
1    Asunafo South      6.581631     -2.543668
2    Asutifi North      7.063915     -2.517689
3    Asutifi South      6.837113     -2.390411
4       Tano North      7.176810     -2.185422
5       Tano South      7.181948     -1.995224
6  Adansi Akrofuom      6.031653     -1.665303
7    Adansi Asokwa      6.175460     -1.433137
8     Adansi North      6.289082     -1.565456
9     Adansi South      6.008318     -1.369419

GUAN entry:
    District  Centroid_Lat  Centroid_Lon
260     Guan        7.2045        0.5191
261     Guan        7.2045        0.5191


In [70]:
# Remove duplicates
fact_table = fact_table.drop_duplicates(subset='District')

print(f"Fact table: {len(fact_table)} districts")

# Check if it's just GUAN that was duplicated
print(f"\nGUAN entries: {len(fact_table[fact_table['District'] == 'Guan'])}")

Fact table: 261 districts

GUAN entries: 1


In [71]:
# The 11 facilities we identified earlier
bad_11 = ['ABUADI CHPS', 'DENYASE CHPS', 'KOKOASE CHPS', 'ATUOBIKROM CHPS', 
          'BESEASE CHPS', 'DOME CHPS', 'NSUOGYASO CHPS',
          'AWULAE BLAY IV CHPS', 'BOATYARD CHPS', 'GONOKROM CHPS', 'KEDZIKOPE CHPS']

# Check which of these are in the 1,030 mismatches
print(f"Are the 11 bad coordinate facilities in the 1,030 mismatches?\n")
for name in bad_11:
    in_wrong = name in wrong_districts['Name'].values
    in_outside = name in facilities_district_check[facilities_district_check['NAME_2'].isna()]['Name'].values
    
    if in_wrong:
        row = wrong_districts[wrong_districts['Name'] == name].iloc[0]
        print(f"  {name:<35} YES — stated: {row['District']:<25} actual: {row['NAME_2']}")
    elif in_outside:
        print(f"  {name:<35} OUTSIDE all districts")
    else:
        print(f"  {name:<35} NO — it matched correctly")

Are the 11 bad coordinate facilities in the 1,030 mismatches?

  ABUADI CHPS                         YES — stated: ADAKLU                    actual: Sekondi Takoradi
  DENYASE CHPS                        YES — stated: TWIFO ATI MORKWA          actual: Upper Denkyira East
  KOKOASE CHPS                        YES — stated: WASSA EAST                actual: Wassa Amenfi West
  ATUOBIKROM CHPS                     YES — stated: KWAHU SOUTH               actual: Kwahu East
  BESEASE CHPS                        YES — stated: AKWAPIM NORTH             actual: Upper Denkyira West
  DOME CHPS                           YES — stated: ASUNAFO NORTH             actual: Sene West
  NSUOGYASO CHPS                      YES — stated: KWAHU AFRAM PLAINS SOUTH  actual: Sene West
  AWULAE BLAY IV CHPS                 OUTSIDE all districts
  BOATYARD CHPS                       OUTSIDE all districts
  GONOKROM CHPS                       OUTSIDE all districts
  KEDZIKOPE CHPS                      OUTSIDE all

In [72]:
# DENYASE CHPS - what do we have now?
denyase = master[master['Name'] == 'DENYASE CHPS'].iloc[0]
print(f"DENYASE CHPS:")
print(f"  Stated District: {denyase['District']}")
print(f"  Coordinates: {denyase['Latitude']}, {denyase['Longitude']}")

# What district does the polygon say?
print(f"\n  Polygon says: Upper Denkyira East")

# Is there a Denyase community in Twifo Ati Morkwa AND in Upper Denkyira East?
# Let's check what other facilities are in Twifo Ati Morkwa
twi_facilities = master[master['District'] == 'TWIFO ATI MORKWA']
print(f"\n  Total facilities in TWIFO ATI MORKWA: {len(twi_facilities)}")
print(f"  DENYASE community: {denyase['Community']}")

DENYASE CHPS:
  Stated District: TWIFO ATI MORKWA
  Coordinates: 5.854229606029756, -1.8013720077775004

  Polygon says: Upper Denkyira East

  Total facilities in TWIFO ATI MORKWA: 19
  DENYASE community: DENYASE 


In [73]:
names = ['DENYASE CHPS', 'KOKOASE CHPS', 'ATUOBIKROM CHPS', 'BESEASE CHPS', 
         'DOME CHPS', 'NSUOGYASO CHPS', 'BOATYARD CHPS', 'GONOKROM CHPS']

print(f"{'Name':<25} {'Current Lat':<20} {'Current Lon':<20} {'District'}")
print("="*85)
for name in names:
    row = master[master['Name'] == name].iloc[0]
    print(f"{name:<25} {row['Latitude']:<20} {row['Longitude']:<20} {row['District']}")

Name                      Current Lat          Current Lon          District
DENYASE CHPS              5.854229606029756    -1.8013720077775004  TWIFO ATI MORKWA
KOKOASE CHPS              5.607065109510294    -2.487296905928909   WASSA EAST
ATUOBIKROM CHPS           6.639825479861686    -0.9623284461520769  KWAHU SOUTH
BESEASE CHPS              6.041993555712988    -2.0105370635976705  AKWAPIM NORTH
DOME CHPS                 7.395566021472596    -0.6246518639064519  ASUNAFO NORTH
NSUOGYASO CHPS            7.624459860903312    -0.49406561717014935 KWAHU AFRAM PLAINS SOUTH
BOATYARD CHPS             5.287440475229086    -0.7292129482592253  GOMOA WEST
GONOKROM CHPS             7.24                 -2.96                DORMAA MUNICIPAL


In [74]:
# Step 3a: Update district labels for the 1,030 mismatched facilities
# Using the polygon's district as the correct one

# Get the facility ID and the polygon's district name for mismatched facilities
updates = facilities_district_check[
    (~facilities_district_check['district_match']) & 
    (facilities_district_check['NAME_2'].notna())
][['ID', 'NAME_2']].copy()

# Show before counts
print(f"BEFORE UPDATE:")
print(f"  Total facilities to update: {len(updates)}")

# Update master dataset - change District to match polygon
for _, row in updates.iterrows():
    master.loc[master['ID'] == row['ID'], 'District'] = row['NAME_2'].upper()

# Verify
print(f"\nAFTER UPDATE:")
print(f"  Sample of updated districts:")
for name in ['DENYASE CHPS', 'BOATYARD CHPS', 'GONOKROM CHPS']:
    d = master[master['Name'] == name].iloc[0]['District']
    print(f"    {name}: {d}")

print(f"\n  Unique districts in master: {master['District'].nunique()}")

BEFORE UPDATE:
  Total facilities to update: 1030

AFTER UPDATE:
  Sample of updated districts:
    DENYASE CHPS: UPPER DENKYIRA EAST
    BOATYARD CHPS: GOMOA WEST
    GONOKROM CHPS: DORMAA MUNICIPAL

  Unique districts in master: 301


In [75]:
# Compare district names before and after
# The polygon uses mixed case like "Weija Gbawe" but we converted to UPPER
# Let's see if that caused duplicates

# Get all unique districts and sort them
all_districts = sorted(master['District'].unique())
print(f"Total unique districts: {len(all_districts)}")

# Look for potential duplicates — same district different formatting
# For example "GA SOUTH" and "Ga South" or "WEIJA GBAWE" and "WEIJA-GBAWE"
print(f"\nSample of district names:")
for d in all_districts[:30]:
    print(f"  '{d}'")

Total unique districts: 301

Sample of district names:
  'ABLEKUMA CENTRAL'
  'ABLEKUMA NORTH'
  'ABLEKUMA WEST'
  'ABUAKWA NORTH'
  'ABUAKWA SOUTH'
  'ABURA-ASEBU-KWAMANKESE'
  'ACCRA'
  'ACCRA METRO'
  'ACHIASE'
  'ADA EAST'
  'ADA WEST'
  'ADAKLU'
  'ADANSI AKROFUOM'
  'ADANSI ASOKWA'
  'ADANSI NORTH'
  'ADANSI SOUTH'
  'ADENTA'
  'ADENTAN'
  'AFADJATO SOUTH'
  'AFADZATO SOUTH'
  'AFIGYA KWABRE NORTH'
  'AFIGYA KWABRE SOUTH'
  'AFIGYA-KWABRE SOUTH'
  'AGONA EAST'
  'AGONA WEST'
  'AGORTIME-ZIOPE'
  'AGOTIME ZIOPE'
  'AHAFO ANO NORTH'
  'AHAFO ANO SOUTH EAST'
  'AHAFO ANO SOUTH WEST'


In [76]:
# Create mapping from old hospital names to GADM polygon names
# We already have district_gadm_mapping which maps our simple_keys to GADM simple_keys
# Now we need to map actual district names

# Build a lookup: simple_key -> GADM actual name
gadm_name_lookup = dict(zip(districts_gdf['simple_key'], districts_gdf['NAME_2'].str.upper()))

# For each facility, get its simple key and map to GADM name
master['simple_key'] = master['District'].apply(make_simple_key)
master['gadm_key'] = master['simple_key'].map(lambda x: district_gadm_mapping.get(x, x))
master['District_new'] = master['gadm_key'].map(gadm_name_lookup)

# Check how many mapped successfully
mapped = master['District_new'].notna().sum()
unmapped = master['District_new'].isna().sum()

print(f"Mapped to GADM name: {mapped}")
print(f"Unmapped: {unmapped}")

if unmapped > 0:
    print(f"\nUnmapped districts:")
    for d in sorted(master[master['District_new'].isna()]['District'].unique()):
        print(f"  {d}")

Mapped to GADM name: 9978
Unmapped: 0


In [77]:
# Replace District with the standardized GADM name
master['District'] = master['District_new']

# Clean up helper columns
master = master.drop(columns=['simple_key', 'gadm_key', 'District_new'])

print(f"Unique districts now: {master['District'].nunique()}")

# Quick sanity check on the ones we know about
print(f"\nSample checks:")
for name in ['DENYASE CHPS', 'BOATYARD CHPS', 'GONOKROM CHPS']:
    d = master[master['Name'] == name].iloc[0]['District']
    print(f"  {name}: {d}")

print(f"\nNo more duplicates like ADENTAN/ADENTA:")
for d in sorted(master['District'].unique()):
    if 'ADENT' in d:
        print(f"  {d}")

Unique districts now: 261

Sample checks:
  DENYASE CHPS: UPPER DENKYIRA EAST
  BOATYARD CHPS: GOMOA WEST
  GONOKROM CHPS: DORMAA

No more duplicates like ADENTAN/ADENTA:
  ADENTA


In [78]:
# Step 3b: Map each facility to its correct district centroid and recalculate distance

# First standardize fact table names to match
fact_table['simple_key'] = fact_table['District'].apply(make_simple_key)

# Standardize master district names to simple_key for matching
master['simple_key'] = master['District'].apply(make_simple_key)

# Check how many match
fact_keys = set(fact_table['simple_key'])
master_keys = set(master['simple_key'])
missing = master_keys - fact_keys

print(f"Master districts matching fact table: {len(master_keys & fact_keys)} of {len(master_keys)}")

if missing:
    print(f"\nMissing from fact table ({len(missing)}):")
    for k in sorted(missing):
        print(f"  {k}")

Master districts matching fact table: 260 of 261

Missing from fact table (1):
  MAMPONG


In [79]:
# What does the fact table call it?
for d in sorted(fact_table['simple_key'].unique()):
    if 'MAMPONG' in d:
        print(f"  Fact table: {d}")

# What does master call it?
for d in sorted(master[master['simple_key'].str.contains('MAMPONG')]['District'].unique()):
    print(f"  Master: {d}")

  Fact table: ASANTE MAMPONG
  Fact table: ASOKORE MAMPONG
  Master: ASOKORE-MAMPONG
  Master: MAMPONG


In [80]:
# Map MAMPONG to ASANTE MAMPONG in fact table lookup
master.loc[master['simple_key'] == 'MAMPONG', 'simple_key'] = 'ASANTE MAMPONG'

# Verify
missing = set(master['simple_key']) - set(fact_table['simple_key'])
print(f"Missing from fact table: {len(missing)}")

Missing from fact table: 0


In [81]:
# Build centroid lookup from fact table
centroid_lookup = dict(zip(
    fact_table['simple_key'], 
    zip(fact_table['Centroid_Lat'], fact_table['Centroid_Lon'])
))

# Update centroids and recalculate distances for ALL facilities
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

updated = 0
for idx, row in master.iterrows():
    key = row['simple_key']
    if key in centroid_lookup:
        clat, clon = centroid_lookup[key]
        master.at[idx, 'Centroid_Lat'] = clat
        master.at[idx, 'Centroid_Lon'] = clon
        dist = haversine(row['Latitude'], row['Longitude'], clat, clon)
        master.at[idx, 'distance_from_centroid_km'] = round(dist, 3)
        updated += 1

# Clean up
master = master.drop(columns=['simple_key'])

print(f"Updated: {updated} facilities")
print(f"Missing centroids: {master['Centroid_Lat'].isna().sum()}")

print(f"\nNew distance stats:")
print(master['distance_from_centroid_km'].describe())

print(f"\nFacilities over 50km from centroid: {len(master[master['distance_from_centroid_km'] > 50])}")
print(f"Facilities over 100km from centroid: {len(master[master['distance_from_centroid_km'] > 100])}")

Updated: 9978 facilities
Missing centroids: 0

New distance stats:
count    9978.000000
mean       11.141601
std         8.952622
min         0.048000
25%         4.662000
50%         9.136500
75%        15.138000
max        99.660000
Name: distance_from_centroid_km, dtype: float64

Facilities over 50km from centroid: 41
Facilities over 100km from centroid: 0


In [82]:
unfound = ['ABUADI CHPS', 'AWULAE BLAY IV CHPS', 'KEDZIKOPE CHPS']

print(f"{'Name':<30} {'District':<25} {'Distance_km'}")
print("="*65)
for name in unfound:
    row = master[master['Name'] == name].iloc[0]
    print(f"{row['Name']:<30} {row['District']:<25} {row['distance_from_centroid_km']}")

Name                           District                  Distance_km
ABUADI CHPS                    SEKONDI TAKORADI          1.986
AWULAE BLAY IV CHPS            ELLEMBELLE                23.841
KEDZIKOPE CHPS                 KETA MUNICIPAL            10.157


In [83]:
far_facilities = master[master['distance_from_centroid_km'] > 50].sort_values('distance_from_centroid_km', ascending=False)

print(f"Facilities over 50km from centroid: {len(far_facilities)}\n")
print(f"{'Name':<40} {'District':<25} {'Region':<15} {'Dist_km'}")
print("="*95)
for _, row in far_facilities.iterrows():
    print(f"{row['Name']:<40} {row['District']:<25} {row['Region']:<15} {row['distance_from_centroid_km']}")

Facilities over 50km from centroid: 41

Name                                     District                  Region          Dist_km
KALIDU CHPS                              CENTRAL GONJA             SAVANNAH        99.66
DASHE CHPS                               NORTH EAST GONJA          SAVANNAH        74.056
JANTONG HEALTH CENTRE                    NORTH EAST GONJA          SAVANNAH        71.283
CHACHE CHPS                              BOLE                      SAVANNAH        70.69
BABATOR CHPS                             BOLE                      SAVANNAH        69.653
JANTONG_KPANDU  CHPS                     NORTH EAST GONJA          SAVANNAH        68.447
BAMBOI HEALTH CENTRE                     BOLE                      SAVANNAH        65.182
SAKPALUA CHPS                            NORTH EAST GONJA          SAVANNAH        64.558
KPALANGASE CHPS                          CENTRAL GONJA             SAVANNAH        63.217
BAMBOI POLYCLINIC                        BOLE                

In [84]:
pd.set_option('display.max_columns', None)
master.head(10)

,ID,Name,Facility_Type,Ownership,Region,District,Sub-District,Community,Latitude,Longitude,has_emonc,has_midwife,has_blood_bank,District_standardized,District_Population,District_Urban_Pop,District_Rural_Pop,Percentage of Urban,Percentage of Rural,Male_Total,Female_Total,Household_Total,NonHousehold_Total,Urban_Male,Urban_Female,Rural_Male,Rural_Female,Urban_Population,Rural_Population,Centroid_Lat,Centroid_Lon,distance_from_centroid_km,bad_coordinates
0,4608,1 MEDICAL RECEPTION STATION(1MRS),POLYCLINIC,QUASI-GOVERNMENT,GREATER ACCRA,KPONE-KATAMANSO,GBETSILE,MICHEL CAMP,5.728982,-0.025255,False,False,False,KPONE KATAMANSO,417334,394882,22452,0.946201,0.053799,208040,209294,416128,1206,196707,198175,11333,11119,394882,22452,5.756751,-0.059359,4.875,False
1,564,2MRS MILITARY HOSPITAL,HOSPITAL,QUASI-GOVERNMENT,WESTERN,EFFIA KWESIMINTSIM,APREMDO,ALREADY BARRACKS,4.914347,-1.805559,False,False,False,EFFIA KWESIMINTSIM MUNICIPAL,173975,173975,-,1.000000,0.000000,85864,88111,170992,2983,85864,88111,-,-,173975,-,4.964357,-1.787798,5.899,False
2,9884,31ST DWM CHPS,CHPS,GOVERNMENT,ASHANTI,KUMASI,MANHYIA -ASH TOWN,ASH TOWN,6.705688,-1.621758,False,False,False,KUMASI METROPOLITAN,443981,443981,-,1.000000,0.000000,213662,230319,413561,30420,213662,230319,-,-,443981,-,6.688186,-1.621228,1.947,False
3,3955,37 MILITARY HOSPITAL,HOSPITAL,QUASI-GOVERNMENT,GREATER ACCRA,AYAWASO EAST,KANDA,<NULL>,5.588470,-0.183260,False,False,False,AYAWASO EAST MUNICIPAL,53004,53004,0,1.000000,0.000000,25438,27566,52508,496,25438,27566,NaN,NaN,53004,NaN,5.590851,-0.191775,0.979,False
4,4865,3E MEDICAL CENTRE,CLINIC,PRIVATE,GREATER ACCRA,WEIJA GBAWE,BORTIANOR,BORTIANOR REDTOP,5.531571,-0.351723,False,False,False,GA SOUTH,350121,266721,83400,0.761797,0.238203,172492,177629,349171,950,131068,135653,41424,41976,266721,83400,5.549603,-0.351312,2.006,False
5,4098,3M&C GHANA LIMITED,HEALTH CENTRE,PRIVATE,GREATER ACCRA,AYAWASO WEST,LEGON,LEGON,5.643030,-0.153326,False,False,False,AYAWASO WEST MUNICIPAL,75303,75303,0,1.000000,0.000000,38614,36689,60952,14351,38614,36689,NaN,NaN,75303,NaN,5.633869,-0.173598,2.464,False
6,7900,3MRS SUNYANI,HOSPITAL,QUASI-GOVERNMENT,BONO,SUNYANI,NEW DORMAA,ASUAKWAH,7.341959,-2.294543,False,False,False,SUNYANI MUNICIPAL,193595,156343,37252,0.807578,0.192422,96358,97237,185031,8564,77088,79255,19270,17982,156343,37252,7.234701,-2.364255,14.190,False
7,10078,3WAY FAMILY CARE CLINIC(CLOSED),CLINIC,PRIVATE,AHAFO,ASUTIFI SOUTH,ACHERENSUA,KROFROM,6.979922,-2.289259,False,False,False,ASUTIFI SOUTH,68394,33236,35158,0.485949,0.514051,34932,33462,66692,1702,16515,16721,18417,16741,33236,35158,6.837113,-2.390411,19.412,False
8,9130,4MRS CLINIC,CLINIC,QUASI-GOVERNMENT,ASHANTI,KUMASI,SUBIN NORTH,4BM BARRACKS,6.694661,-1.629465,False,False,False,KUMASI METROPOLITAN,443981,443981,-,1.000000,0.000000,213662,230319,413561,30420,213662,230319,-,-,443981,-,6.688186,-1.621228,1.160,False
9,3780,64 BENCH CHPS,CHPS,GOVERNMENT,NORTHERN,TAMALE,TAMALE CENTRAL,CHANGLI,9.395928,-0.838161,False,False,False,TAMALE METROPOLITAN,374744,374744,-,1.000000,0.000000,185051,189693,365510,9234,185051,189693,-,-,374744,-,9.373733,-0.743034,10.724,False


In [85]:
master = master.drop(columns=['bad_coordinates'])
print(f"✅ Removed bad_coordinates column")
print(f"Master dataset: {master.shape[0]} facilities x {master.shape[1]} columns")

✅ Removed bad_coordinates column
Master dataset: 9978 facilities x 32 columns


In [86]:
master.to_csv(r"C:\Users\hp\OneDrive\Documents\That Tech Girlie Passion Projects\Ghana Heathcare Project\master_dataset_v2.csv", index=False)

print("✅ Master dataset v2 saved!")

✅ Master dataset v2 saved!


In [87]:
# end of data cleaning and preparation phase 2